# Alpine Ski Motion Analysis Development Notebook

This notebook contains my actual development process, including debugging, iterations, and the moments where I discovered my initial assumptions were wrong. The commentary cells document my reasoning at each key decision point.

**Note on AI assistance:** I used an AI assistant for coding support and as a thinking partner, while the core direction — identifying problems, questioning assumptions, and choosing the honest confidence-aware design — was driven by my own reasoning.

In [ ]:
!pip install -U ultralytics
!yolo checks

In [ ]:
import os, glob, zipfile, json
from pathlib import Path

# 1) 确保 Drive 已挂载
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

# 2) 在 /content 和整个 Drive 里找 ski2dpose 的 zip,解压到 /content
#    (图片那个 86MB,解压约需几十秒,正常)
found = glob.glob('/content/*.zip') + glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
for z in found:
    if 'ski2dpose' in os.path.basename(z).lower():
        print('解压:', z)
        with zipfile.ZipFile(z) as f:
            f.extractall('/content')

ROOT = Path('/content')

# 3) 定位标签文件
labels_json = next(ROOT.rglob('ski2dpose_labels.json'))
print('找到标签:', labels_json)

# 4) 建立 (video, split, img) -> webp 路径 索引
webp_index = {}
for p in ROOT.rglob('*.webp'):
    webp_index[(p.parent.parent.name, p.parent.name, p.stem)] = p
print('找到 webp 图片:', len(webp_index), '张')

# 5) 验证 JSON 结构
with open(labels_json) as f:
    labels = json.load(f)
vid = next(iter(labels)); sid = next(iter(labels[vid])); iid = next(iter(labels[vid][sid]))
rec = labels[vid][sid][iid]
print('示例 key:', vid, sid, iid)
print('annotation 点数:', len(rec['annotation']), ' 前2点:', rec['annotation'][:2])

In [ ]:
import shutil
from pathlib import Path

OUT = Path('/content/ski_yolo')
for sub in ['images/train','images/val','labels/train','labels/val']:
    (OUT/sub).mkdir(parents=True, exist_ok=True)

VAL_SPLITS = {('5UHRvqx1iuQ','0'),('5UHRvqx1iuQ','1'),
              ('oKQFABiOTw8','0'),('oKQFABiOTw8','1'),('oKQFABiOTw8','2'),
              ('qxfgw1Kd98A','0'),('qxfgw1Kd98A','1'),
              ('uLW74013Wp0','0'),('uLW74013Wp0','1'),
              ('zW1bF2PsB0M','0'),('zW1bF2PsB0M','1')}

PAD = 0.08
c = lambda v: min(1.0, max(0.0, v))   # clip 到 [0,1]

n_tr=n_va=n_miss=0; sample=None
for vid, splits in labels.items():
    for sid, imgs in splits.items():
        is_val = (vid,sid) in VAL_SPLITS
        for iid, rec in imgs.items():
            key=(vid,sid,iid)
            if key not in webp_index: n_miss+=1; continue
            ann = rec['annotation']                       # 24 x [x,y,vis]
            xs=[c(p[0]) for p in ann]; ys=[c(p[1]) for p in ann]
            xmin,xmax,ymin,ymax=min(xs),max(xs),min(ys),max(ys)
            cx=c((xmin+xmax)/2); cy=c((ymin+ymax)/2)
            w=c((xmax-xmin)*(1+2*PAD)); h=c((ymax-ymin)*(1+2*PAD))
            kp=[]
            for x,y,v in ann:                             # 1->2(可见), 0->1(遮挡)
                kp += [f'{c(x):.6f}', f'{c(y):.6f}', '2' if int(v)==1 else '1']
            line=f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} ' + ' '.join(kp)
            split='val' if is_val else 'train'
            stem=iid                                       # iid 已全局唯一,直接用
            shutil.copy(webp_index[key], OUT/f'images/{split}/{stem}.webp')
            (OUT/f'labels/{split}/{stem}.txt').write_text(line+'\n')
            if is_val: n_va+=1
            else: n_tr+=1
            if sample is None: sample=line

print(f'✅ 转换完成  train={n_tr}  val={n_va}  缺失={n_miss}')
print('示例标签:', sample[:120], '...')

yaml_text = f"""path: {OUT}
train: images/train
val: images/val

kpt_shape: [24, 3]
flip_idx: [0, 1, 6, 7, 8, 9, 2, 3, 4, 5, 13, 14, 15, 10, 11, 12, 20, 21, 22, 23, 16, 17, 18, 19]

names:
  0: skier
"""
(OUT/'ski-pose.yaml').write_text(yaml_text)
print('已写出配置:\n', yaml_text)

In [ ]:
import torch
from ultralytics import YOLO

print('GPU:', torch.cuda.get_device_name(0))

model = YOLO('yolo26n-pose.pt')

results = model.train(
    data='/content/ski_yolo/ski-pose.yaml',
    epochs=100,
    imgsz=960,
    batch=16,
    patience=20,
    project='/content/drive/MyDrive/ski_app/runs',
    name='ski_y26n_v1',
    resume=False,
    pose=12.0,
    plots=True,          # 显式开启:自动画 loss/PR/F1/混淆矩阵
)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

b = '/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best.pt'
l = '/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/last.pt'
print('best.pt:', os.path.exists(b), '| last.pt:', os.path.exists(l))

In [ ]:
from pathlib import Path
from IPython.display import Image, display

run = Path('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1')
for img in ['results.png', 'PosePR_curve.png', 'PoseF1_curve.png', 'confusion_matrix.png']:
    p = run / img
    print('=====', img, '=====')
    display(Image(filename=str(p))) if p.exists() else print('没找到', img)

In [ ]:
from pathlib import Path
run = Path('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1')
print('目录存在:', run.exists())
for f in sorted(run.rglob('*')):
    print(f.relative_to(run), '|', round(f.stat().st_size/1024,1), 'KB')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(run / 'results.csv')
df.columns = [c.strip() for c in df.columns]
print('总轮数:', len(df))
print('列名:', list(df.columns))

fig, axes = plt.subplots(1, 2, figsize=(14,5))

# 左图:关键点精度(早停的真正依据)
for col in ['metrics/mAP50(P)', 'metrics/mAP50-95(P)']:
    if col in df.columns:
        axes[0].plot(df['epoch'], df[col], label=col)
axes[0].set_title('Pose mAP (越高越好)'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(True)

# 右图:训练 vs 验证 的 pose loss(看过拟合)
for col in ['train/pose_loss', 'val/pose_loss']:
    if col in df.columns:
        axes[1].plot(df['epoch'], df[col], label=col)
axes[1].set_title('Pose Loss (越低越好)'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout(); plt.show()

# 打印最佳那一轮
best_col = 'metrics/mAP50-95(P)' if 'metrics/mAP50-95(P)' in df.columns else df.columns[-1]
best_row = df.loc[df[best_col].idxmax()]
print(f'\n最佳轮: epoch {int(best_row["epoch"])} | pose mAP50-95 = {best_row[best_col]:.4f}')

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best.pt')

In [ ]:
import torch
print('CUDA 可用:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '没有GPU')

In [ ]:
import glob, zipfile
from google.colab import drive
drive.mount('/content/drive')
for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
    if 'ski2dpose' in z.lower():
        with zipfile.ZipFile(z) as f: f.extractall('/content')
print('数据已解压')

In [ ]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/last.pt')
model.train(resume=True)

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
run = '/content/drive/MyDrive/ski_app/runs/ski_y26n_v1'
for f in ['weights/best.pt','results.png','PosePR_curve.png','PoseF1_curve.png','confusion_matrix.png']:
    p = os.path.join(run, f)
    print('✅' if os.path.exists(p) else '❌', f)

In [ ]:
# Spike-1:用 best.pt 跑验证图,可视化24个关键点
import glob, random
from pathlib import Path
from ultralytics import YOLO
from IPython.display import Image, display

# 1) 加载你训练好的模型
model = YOLO('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best.pt')

# 2) 找验证图(如果临时盘清空了,先解压恢复)
val_imgs = glob.glob('/content/ski_yolo/images/val/*.webp')
if len(val_imgs) == 0:
    import zipfile
    print('验证图不在,重新解压数据...')
    for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
        if 'ski2dpose' in z.lower():
            with zipfile.ZipFile(z) as f: f.extractall('/content')
    # 直接从解压出的原图里挑(val 那几个视频)
    val_imgs = glob.glob('/content/**/5UHRvqx1iuQ/**/*.webp', recursive=True)
print('找到验证图:', len(val_imgs), '张')

# 3) 随机挑4张跑预测
picks = random.sample(val_imgs, min(4, len(val_imgs)))
results = model.predict(picks, imgsz=960, conf=0.5,
                        save=True, project='/content', name='spike1', exist_ok=True)

# 4) 显示画好点的结果图
out = sorted(glob.glob('/content/spike1/*.webp') + glob.glob('/content/spike1/*.jpg'))
for p in out:
    print('=====', Path(p).name, '=====')
    display(Image(filename=p))

In [ ]:
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best.pt')

# 导出 TFLite(FP32 先跑通,确认算子兼容;INT8量化下一步再做)
model.export(format='tflite', imgsz=960)

In [ ]:
from ultralytics import YOLO
import glob, random
from pathlib import Path
from IPython.display import Image, display

# 加载导出的 TFLite 模型
tflite_model = YOLO('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best_saved_model/best_float32.tflite', task='pose')

# 拿验证图,一张一张跑(tflite 固定 batch=1)
val_imgs = glob.glob('/content/**/5UHRvqx1iuQ/**/*.webp', recursive=True)
picks = random.sample(val_imgs, 2)

for img in picks:                          # ← 关键:逐张循环
    tflite_model.predict(img, imgsz=960, conf=0.5,
                         save=True, project='/content', name='spike2_tflite', exist_ok=True)

# 显示结果
for p in sorted(glob.glob('/content/spike2_tflite/*')):
    print('=====', Path(p).name, '=====')
    display(Image(filename=p))

In [ ]:
# 1. 重装 ultralytics
!pip install -U ultralytics

# 2. 挂载 Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. 加载模型确认
from ultralytics import YOLO
model = YOLO('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best.pt')
print('✅ 模型加载成功,准备进第三关')

In [ ]:
import json, glob, zipfile
import numpy as np
import matplotlib.pyplot as plt

# ---------- 1) 定位标签 JSON(临时盘清空了就从Drive重解压) ----------
labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)
if not labels_path:
    for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
        if 'labels' in z.lower() and 'ski2dpose' in z.lower():
            with zipfile.ZipFile(z) as f: f.extractall('/content')
    labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)
labels = json.load(open(labels_path[0]))
print('加载标签,视频数:', len(labels))

# ---------- 2) 关键点索引(你的24点方案) ----------
SH_R, SH_L = 2, 6      # 右肩/左肩
HIP_R, HIP_L = 10, 13  # 右髋/左髋
TOE_R, TOE_L = 17, 21  # 右脚尖/左脚尖(雪板)
EPS = 1e-6

def visible(pt):   # pt = [x, y, v];v=1可见
    return pt[2] == 1

R_shoulder, R_ski = [], []

# ---------- 3) 遍历全部图,算两个视角指标 ----------
for vid, splits in labels.items():
    for sid, imgs in splits.items():
        for iid, rec in imgs.items():
            ann = rec['annotation']  # 24 x [x,y,v]
            # 躯干高:肩中点y 到 髋中点y(需要肩髋至少各一侧可见)
            sh_pts  = [ann[i] for i in (SH_R, SH_L) if visible(ann[i])]
            hip_pts = [ann[i] for i in (HIP_R, HIP_L) if visible(ann[i])]
            if not sh_pts or not hip_pts:
                continue
            sh_y  = np.mean([p[1] for p in sh_pts])
            hip_y = np.mean([p[1] for p in hip_pts])
            torso_h = abs(sh_y - hip_y) + EPS
            if torso_h < 0.02:   # 躯干太短,姿态异常,跳过
                continue

            # 指标1:肩宽/躯干高(需左右肩都可见)
            if visible(ann[SH_R]) and visible(ann[SH_L]):
                R_shoulder.append(abs(ann[SH_R][0] - ann[SH_L][0]) / torso_h)
            # 指标2:雪板间距/躯干高(需左右脚尖都可见)
            if visible(ann[TOE_R]) and visible(ann[TOE_L]):
                R_ski.append(abs(ann[TOE_R][0] - ann[TOE_L][0]) / torso_h)

R_shoulder, R_ski = np.array(R_shoulder), np.array(R_ski)

# ---------- 4) 统计 + 直方图 ----------
def stats(name, arr):
    q = np.percentile(arr, [10,25,50,75,90])
    print(f'{name}: 样本={len(arr)} | 中位数={q[2]:.3f} | '
          f'25%~75%=[{q[1]:.3f}, {q[3]:.3f}] | 10%~90%=[{q[0]:.3f}, {q[4]:.3f}]')

print('\n===== 视角指标统计(值越大=越偏正/背面,越小=越偏侧面) =====')
stats('肩宽指标 R_shoulder', R_shoulder)
stats('雪板指标 R_ski      ', R_ski)

fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].hist(R_shoulder, bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('R_shoulder (shoulder width / torso height)')
axes[0].set_xlabel('small=side view   |   large=front/back view')
axes[1].hist(R_ski, bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('R_ski (ski separation / torso height)')
axes[1].set_xlabel('small=side view   |   large=back view')
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np

def joint_angle(p1, p2, p3, vis_thresh=1, eps=1e-8):
    """
    计算关节夹角(顶点在 p2)。
    输入三个关键点,每个是 [x, y, v]:x,y 坐标,v 可见性(1=可见,0=遮挡)。
    例:算膝关节 → p1=髋, p2=膝(顶点), p3=踝

    返回 (角度, 是否可信):
      - 角度:0~180 度的 2D 投影夹角
      - 是否可信:三个点都达到可见性阈值才为 True
    """
    p1, p2, p3 = np.asarray(p1, float), np.asarray(p2, float), np.asarray(p3, float)

    # —— 保护①:可见性检查(视角不好/遮挡时,点不可信)——
    trustworthy = (p1[2] >= vis_thresh and p2[2] >= vis_thresh and p3[2] >= vis_thresh)

    # —— 从顶点 p2 引出两个向量 ——
    a = p1[:2] - p2[:2]   # 膝→髋
    b = p3[:2] - p2[:2]   # 膝→踝

    # —— 保护②:防除零(两点重合时模长为0)——
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < eps or nb < eps:
        return np.nan, False

    # —— 点积公式求夹角 ——
    cos_theta = np.dot(a, b) / (na * nb)

    # —— 保护③:clip 防止浮点越界导致 arccos 出 NaN ——
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    angle = np.degrees(np.arccos(cos_theta))
    return angle, trustworthy

In [ ]:
import json, glob

# 关键点索引(你的24点方案)
HIP_R, KNEE_R, ANKLE_R = 10, 11, 12   # 右髋/右膝/右踝
HIP_L, KNEE_L, ANKLE_L = 13, 14, 15

# 随便取一张图的标注
labels = json.load(open(glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]))
vid = next(iter(labels)); sid = next(iter(labels[vid])); iid = next(iter(labels[vid][sid]))
ann = labels[vid][sid][iid]['annotation']

# 算左右膝关节角度
ang_r, ok_r = joint_angle(ann[HIP_R], ann[KNEE_R], ann[ANKLE_R])
ang_l, ok_l = joint_angle(ann[HIP_L], ann[KNEE_L], ann[ANKLE_L])

print(f'右膝角度: {ang_r:.1f}°  可信:{ok_r}')
print(f'左膝角度: {ang_l:.1f}°  可信:{ok_l}')

In [ ]:
# ===== 全身关节角度计算 =====
# 复用上面已验证的 joint_angle(p1, p2, p3) 函数

# ---- 24点索引表(你的方案)----
HEAD, NECK = 0, 1
SH_R, EL_R, WR_R = 2, 3, 4          # 右肩/右肘/右腕(hand)
SH_L, EL_L, WR_L = 6, 7, 8          # 左肩/左肘/左腕
HIP_R, KNEE_R, ANK_R = 10, 11, 12   # 右髋/右膝/右踝
HIP_L, KNEE_L, ANK_L = 13, 14, 15   # 左髋/左膝/左踝

def all_joint_angles(ann):
    """
    输入:一张图的24点标注/预测,每点 [x,y,v]
    输出:dict { 角度名: (角度值, 是否可信) }
    """
    # 每个角度 = (名称, 点1, 顶点, 点3)
    joint_defs = [
        ('右膝', HIP_R,  KNEE_R, ANK_R),   # 髋-膝-踝:屈膝程度
        ('左膝', HIP_L,  KNEE_L, ANK_L),
        ('右髋', SH_R,   HIP_R,  KNEE_R),   # 肩-髋-膝:身体折叠
        ('左髋', SH_L,   HIP_L,  KNEE_L),
        ('右肘', SH_R,   EL_R,   WR_R),     # 肩-肘-腕:手臂弯曲
        ('左肘', SH_L,   EL_L,   WR_L),
    ]

    angles = {}
    for name, i1, i2, i3 in joint_defs:
        ang, ok = joint_angle(ann[i1], ann[i2], ann[i3])
        angles[name] = (ang, ok)
    return angles


# ---- 测试:拿一张图算全身角度 ----
import json, glob
labels = json.load(open(glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]))
vid = next(iter(labels)); sid = next(iter(labels[vid])); iid = next(iter(labels[vid][sid]))
ann = labels[vid][sid][iid]['annotation']

print('===== 全身关节角度 =====')
for name, (ang, ok) in all_joint_angles(ann).items():
    flag = '✅可信' if ok else '⚠️含遮挡点'
    val = f'{ang:6.1f}°' if not np.isnan(ang) else '  N/A '
    print(f'{name}: {val}  {flag}')

In [ ]:
import numpy as np

def body_line_tilt(p_low, p_high, vis_thresh=1, eps=1e-8):
    """
    算一条身体线相对【图像垂直方向】的带符号倾斜角。
    p_low  = 线的下端点 [x,y,v](如髋中点)
    p_high = 线的上端点 [x,y,v](如肩中点)

    返回 (角度, 是否可信):
      角度:带符号,单位度
        0°   = 完全竖直(在图像里竖着)
        +    = 上端偏向右(肩在髋右边 → 身体向右倾)
        -    = 上端偏向左(身体向左倾)
        ±90° = 完全水平
      可信:两个端点都可见才 True
    """
    p_low  = np.asarray(p_low, float)
    p_high = np.asarray(p_high, float)

    # 可见性检查
    trustworthy = (p_low[2] >= vis_thresh and p_high[2] >= vis_thresh)

    # 身体线向量:下端 → 上端
    vx = p_high[0] - p_low[0]
    vy = p_high[1] - p_low[1]   # 注意:y轴朝下

    # 防除零(两端点重合)
    if (vx*vx + vy*vy) ** 0.5 < eps:
        return np.nan, False

    # —— 关键:y轴朝下的处理 ——
    # 图像里"上方"是 y 减小方向。竖直站立时 vy<0(上端在上)。
    # 用 atan2 算"偏离竖直"的带符号角度:
    #   水平分量 vx(左右偏移)
    #   竖直分量 用 -vy(把"图像向上"翻成数学正向)
    # atan2(vx, -vy):vx>0(上端偏右)→ 正角度;vx<0 → 负角度
    angle = np.degrees(np.arctan2(vx, -vy))

    return angle, trustworthy

In [ ]:
# 索引
SH_R, SH_L = 2, 6
HIP_R, HIP_L = 10, 13
KNEE_R, KNEE_L = 11, 14
ANK_R, ANK_L = 12, 15

def midpoint(pa, pb):
    """两点中点,可见性取两者最小(都可见才算可见)"""
    pa, pb = np.asarray(pa, float), np.asarray(pb, float)
    return np.array([(pa[0]+pb[0])/2, (pa[1]+pb[1])/2, min(pa[2], pb[2])])

# 取一张图
import json, glob
labels = json.load(open(glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]))
vid = next(iter(labels)); sid = next(iter(labels[vid])); iid = next(iter(labels[vid][sid]))
ann = labels[vid][sid][iid]['annotation']

# 躯干线:髋中点 → 肩中点
hip_mid = midpoint(ann[HIP_R], ann[HIP_L])
sh_mid  = midpoint(ann[SH_R],  ann[SH_L])
trunk_tilt, ok_t = body_line_tilt(hip_mid, sh_mid)

# 大腿线(右):膝 → 髋
thigh_r_tilt, ok_tr = body_line_tilt(ann[KNEE_R], ann[HIP_R])
# 小腿线(右):踝 → 膝
shank_r_tilt, ok_sr = body_line_tilt(ann[ANK_R], ann[KNEE_R])

print('===== 身体线倾斜角(相对图像垂直,+右倾 -左倾)=====')
print(f'躯干:   {trunk_tilt:+6.1f}°  {"✅" if ok_t else "⚠️"}')
print(f'右大腿: {thigh_r_tilt:+6.1f}°  {"✅" if ok_tr else "⚠️"}')
print(f'右小腿: {shank_r_tilt:+6.1f}°  {"✅" if ok_sr else "⚠️"}')

In [ ]:
def angulation(p_hip, p_knee, p_shoulder_mid, p_hip_mid, vis_thresh=1):
    """
    算反弓角(angulation)= 大腿内倾 - 躯干内倾。
    参数:
      p_hip, p_knee   : 某一侧的髋、膝(算大腿线:膝→髋)
      p_shoulder_mid  : 肩中点(算躯干线:髋中点→肩中点)
      p_hip_mid       : 髋中点
    返回 (反弓角, 是否可信):
      ≈0   : 身体像直棍,无反弓(新手倾向)
      较大 : 下半身倒、上半身正,强反弓(高手倾向)
      符号 : 反映折向(配合内倾方向解读)
    """
    # 大腿内倾(膝→髋)
    thigh_tilt, ok_thigh = body_line_tilt(p_knee, p_hip, vis_thresh)
    # 躯干内倾(髋中点→肩中点)
    trunk_tilt, ok_trunk = body_line_tilt(p_hip_mid, p_shoulder_mid, vis_thresh)

    if np.isnan(thigh_tilt) or np.isnan(trunk_tilt):
        return np.nan, False

    angu = thigh_tilt - trunk_tilt          # 核心:差值抵消参考方向
    trustworthy = ok_thigh and ok_trunk     # 两条线都可信才可信
    return angu, trustworthy


# ---- 测试 ----
hip_mid = midpoint(ann[HIP_R], ann[HIP_L])
sh_mid  = midpoint(ann[SH_R],  ann[SH_L])

# 用右腿算反弓
angu_r, ok_r = angulation(ann[HIP_R], ann[KNEE_R], sh_mid, hip_mid)
# 用左腿算反弓
angu_l, ok_l = angulation(ann[HIP_L], ann[KNEE_L], sh_mid, hip_mid)

# 同时打印各分量,方便理解
trunk_tilt, _   = body_line_tilt(hip_mid, sh_mid)
thigh_r, _      = body_line_tilt(ann[KNEE_R], ann[HIP_R])
thigh_l, _      = body_line_tilt(ann[KNEE_L], ann[HIP_L])

print('===== 反弓分析 =====')
print(f'躯干内倾:     {trunk_tilt:+6.1f}°')
print(f'右大腿内倾:   {thigh_r:+6.1f}°   →  右侧反弓: {angu_r:+6.1f}°  {"✅" if ok_r else "⚠️"}')
print(f'左大腿内倾:   {thigh_l:+6.1f}°   →  左侧反弓: {angu_l:+6.1f}°  {"✅" if ok_l else "⚠️"}')

In [ ]:
import numpy as np

# 身体段定义:(名称, 端点A索引, 端点B索引, 质量比例)
# 质心 = 两端点中点;质量来自 Dempster 简化系数
SEGMENTS = [
    ('head',     0, 1, 0.08),    # 头颈:head-neck
    ('trunk',  None, None, 0.50), # 躯干:肩中点-髋中点(特殊,下面处理)
    ('uarm_R',   2, 3, 0.027),   # 右上臂:肩-肘
    ('uarm_L',   6, 7, 0.027),   # 左上臂
    ('farm_R',   3, 4, 0.022),   # 右前臂+手:肘-腕
    ('farm_L',   7, 8, 0.022),   # 左前臂+手
    ('thigh_R', 10,11, 0.10),    # 右大腿:髋-膝
    ('thigh_L', 13,14, 0.10),    # 左大腿
    ('shank_R', 11,12, 0.06),    # 右小腿+脚:膝-踝
    ('shank_L', 14,15, 0.06),    # 左小腿+脚
]
SH_R, SH_L, HIP_R, HIP_L = 2, 6, 10, 13

def center_of_mass(ann, vis_thresh=1):
    """
    用分段质量模型算2D重心。
    返回 (com_x, com_y, 覆盖的质量比例, 是否足够可信)
      覆盖质量比例:实际用上的身体段质量和(1.0=全部可用)
      可信:覆盖质量足够高(如>0.7)才算可信
    """
    total_mass = 0.0
    weighted_x, weighted_y = 0.0, 0.0

    for name, ia, ib, mass in SEGMENTS:
        # 躯干特殊:用肩中点和髋中点
        if name == 'trunk':
            sh  = [(ann[SH_R][k]+ann[SH_L][k])/2 for k in range(3)]
            hip = [(ann[HIP_R][k]+ann[HIP_L][k])/2 for k in range(3)]
            pa, pb = sh, hip
        else:
            pa, pb = ann[ia], ann[ib]

        # 两端点都可见才用这段
        if pa[2] < vis_thresh or pb[2] < vis_thresh:
            continue

        # 段质心 = 两端中点
        cx = (pa[0] + pb[0]) / 2
        cy = (pa[1] + pb[1]) / 2

        weighted_x += mass * cx
        weighted_y += mass * cy
        total_mass += mass

    if total_mass < 1e-6:
        return np.nan, np.nan, 0.0, False

    # 归一化(只除以实际用到的质量,补偿缺失段)
    com_x = weighted_x / total_mass
    com_y = weighted_y / total_mass
    trustworthy = (total_mass >= 0.70)   # 至少覆盖70%质量才可信
    return com_x, com_y, total_mass, trustworthy


# ---- 测试 ----
cx, cy, mass_cov, ok = center_of_mass(ann)
print('===== 重心(CoM)=====')
print(f'重心坐标: ({cx:.3f}, {cy:.3f})  (归一化图像坐标)')
print(f'覆盖质量: {mass_cov*100:.0f}%   可信: {"✅" if ok else "⚠️"}')

In [ ]:
%%writefile /content/drive/MyDrive/ski_app/ski_physics.py
"""
ski_physics.py — 双板滑雪姿态物理量计算
输入:24个关键点 [x, y, v]  (x,y归一化或像素均可;v: 1=可见, 0=遮挡)
作者备注:所有"绝对角度"假设相机大致端平;2D量受透视影响,详见各函数说明。
"""
import numpy as np

# ========== 24点索引 ==========
HEAD, NECK = 0, 1
SH_R, EL_R, WR_R = 2, 3, 4          # 右肩/肘/腕
PB_R = 5                             # 右杖篮
SH_L, EL_L, WR_L = 6, 7, 8          # 左肩/肘/腕
PB_L = 9                             # 左杖篮
HIP_R, KNEE_R, ANK_R = 10, 11, 12   # 右髋/膝/踝
HIP_L, KNEE_L, ANK_L = 13, 14, 15   # 左髋/膝/踝
SKI_TIP_R, TOE_R, HEEL_R, SKI_TAIL_R = 16, 17, 18, 19   # 右板尖/脚尖/脚跟/板尾
SKI_TIP_L, TOE_L, HEEL_L, SKI_TAIL_L = 20, 21, 22, 23   # 左板

# ========== 工具 ==========
def midpoint(pa, pb):
    """两点中点,可见性取两者最小"""
    pa, pb = np.asarray(pa, float), np.asarray(pb, float)
    return np.array([(pa[0]+pb[0])/2, (pa[1]+pb[1])/2, min(pa[2], pb[2])])


# ========== A: 关节夹角 ==========
def joint_angle(p1, p2, p3, vis_thresh=1, eps=1e-8):
    """顶点在p2的夹角(0~180°)。返回(角度, 可信)。三层保护:可见性/防除零/clip。"""
    p1, p2, p3 = np.asarray(p1,float), np.asarray(p2,float), np.asarray(p3,float)
    trustworthy = (p1[2]>=vis_thresh and p2[2]>=vis_thresh and p3[2]>=vis_thresh)
    a = p1[:2] - p2[:2]
    b = p3[:2] - p2[:2]
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < eps or nb < eps:
        return np.nan, False
    cos_theta = np.clip(np.dot(a, b)/(na*nb), -1.0, 1.0)
    return float(np.degrees(np.arccos(cos_theta))), trustworthy


def all_joint_angles(ann, vis_thresh=1):
    """一次算全身关节角。返回 {名称: (角度, 可信)}。"""
    defs = [
        ('右膝', HIP_R, KNEE_R, ANK_R), ('左膝', HIP_L, KNEE_L, ANK_L),
        ('右髋', SH_R,  HIP_R,  KNEE_R),('左髋', SH_L,  HIP_L,  KNEE_L),
        ('右肘', SH_R,  EL_R,   WR_R),  ('左肘', SH_L,  EL_L,   WR_L),
    ]
    return {name: joint_angle(ann[i1], ann[i2], ann[i3], vis_thresh)
            for name, i1, i2, i3 in defs}


# ========== C: 身体线倾斜 & 反弓 ==========
def body_line_tilt(p_low, p_high, vis_thresh=1, eps=1e-8):
    """身体线相对图像垂直方向的带符号倾斜角。0°=竖直, +上端偏右, -上端偏左。
       注意:y轴朝下;'相对重力'仅在相机端平时成立。返回(角度, 可信)。"""
    p_low, p_high = np.asarray(p_low,float), np.asarray(p_high,float)
    trustworthy = (p_low[2]>=vis_thresh and p_high[2]>=vis_thresh)
    vx = p_high[0] - p_low[0]
    vy = p_high[1] - p_low[1]
    if (vx*vx + vy*vy)**0.5 < eps:
        return np.nan, False
    return float(np.degrees(np.arctan2(vx, -vy))), trustworthy


def angulation(p_hip, p_knee, p_sh_mid, p_hip_mid, vis_thresh=1):
    """反弓角 = 大腿内倾 - 躯干内倾。对相机倾斜/坡度/前后镜像鲁棒。返回(角度, 可信)。"""
    thigh, ok_t = body_line_tilt(p_knee, p_hip, vis_thresh)
    trunk, ok_r = body_line_tilt(p_hip_mid, p_sh_mid, vis_thresh)
    if np.isnan(thigh) or np.isnan(trunk):
        return np.nan, False
    return float(thigh - trunk), (ok_t and ok_r)


# ========== B: 重心 ==========
# (名称, 端点A, 端点B, 质量比例);trunk特殊处理。系数来自Dempster简化值。
_SEGMENTS = [
    ('head', HEAD, NECK, 0.08), ('trunk', None, None, 0.50),
    ('uarm_R', SH_R, EL_R, 0.027), ('uarm_L', SH_L, EL_L, 0.027),
    ('farm_R', EL_R, WR_R, 0.022), ('farm_L', EL_L, WR_L, 0.022),
    ('thigh_R', HIP_R, KNEE_R, 0.10), ('thigh_L', HIP_L, KNEE_L, 0.10),
    ('shank_R', KNEE_R, ANK_R, 0.06), ('shank_L', KNEE_L, ANK_L, 0.06),
]

def center_of_mass(ann, vis_thresh=1, min_mass=0.70):
    """分段质量模型算2D重心。返回 (com_x, com_y, 覆盖质量, 可信)。
       缺失段跳过并重新归一化;覆盖质量<min_mass则不可信。"""
    tot, wx, wy = 0.0, 0.0, 0.0
    for name, ia, ib, mass in _SEGMENTS:
        if name == 'trunk':
            pa = midpoint(ann[SH_R], ann[SH_L])
            pb = midpoint(ann[HIP_R], ann[HIP_L])
        else:
            pa, pb = np.asarray(ann[ia],float), np.asarray(ann[ib],float)
        if pa[2] < vis_thresh or pb[2] < vis_thresh:
            continue
        wx += mass * (pa[0]+pb[0])/2
        wy += mass * (pa[1]+pb[1])/2
        tot += mass
    if tot < 1e-6:
        return np.nan, np.nan, 0.0, False
    return wx/tot, wy/tot, tot, (tot >= min_mass)


# ========== 一键提取所有物理量 ==========
def extract_all(ann, vis_thresh=1):
    """对一帧的24点,算出所有物理量,打包成dict。"""
    sh_mid  = midpoint(ann[SH_R], ann[SH_L])
    hip_mid = midpoint(ann[HIP_R], ann[HIP_L])
    out = {}
    out['joints'] = all_joint_angles(ann, vis_thresh)
    out['trunk_tilt'] = body_line_tilt(hip_mid, sh_mid, vis_thresh)
    out['angulation_R'] = angulation(ann[HIP_R], ann[KNEE_R], sh_mid, hip_mid, vis_thresh)
    out['angulation_L'] = angulation(ann[HIP_L], ann[KNEE_L], sh_mid, hip_mid, vis_thresh)
    cx, cy, mcov, ok = center_of_mass(ann, vis_thresh)
    out['com'] = (cx, cy, mcov, ok)
    return out

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ski_app')   # 让python找到模块

import importlib, ski_physics
importlib.reload(ski_physics)   # 确保用最新版
from ski_physics import extract_all

# 拿之前那张图测,确认一键提取所有物理量
result = extract_all(ann)
print('===== 一键提取所有物理量 =====')
for name, (ang, ok) in result['joints'].items():
    print(f'  {name}: {ang:6.1f}°  {"✅" if ok else "⚠️"}')
print(f"躯干内倾: {result['trunk_tilt'][0]:+.1f}°  {'✅' if result['trunk_tilt'][1] else '⚠️'}")
print(f"右反弓:   {result['angulation_R'][0]:+.1f}°  {'✅' if result['angulation_R'][1] else '⚠️'}")
print(f"左反弓:   {result['angulation_L'][0]:+.1f}°  {'✅' if result['angulation_L'][1] else '⚠️'}")
cx, cy, mcov, ok = result['com']
print(f"重心:     ({cx:.3f}, {cy:.3f})  覆盖{mcov*100:.0f}%  {'✅' if ok else '⚠️'}")

### Checking which metrics I can actually trust
Before building analysis, I want to know whether the model's predicted keypoints are truly reliable. I compare metrics computed from model predictions against those from true annotations and labels on the same frame.

In [ ]:
import sys, glob, json
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics
importlib.reload(ski_physics)
from ski_physics import extract_all
from ultralytics import YOLO
import numpy as np

# 1) 加载模型
model = YOLO('/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best.pt')

# 2) 找一张验证图 + 它的标注。先定位图片路径
labels = json.load(open(glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]))
# 取第一个有图的记录
for vid in labels:
    for sid in labels[vid]:
        for iid in labels[vid][sid]:
            rec = labels[vid][sid][iid]
            gt_ann = rec['annotation']   # 标注点(24 x [x,y,v])
            target_iid = iid
            break
        break
    break

# 找这张图的实际文件
img_path = glob.glob(f'/content/**/{target_iid}*.webp', recursive=True)
if not img_path:
    # 临时盘没图就重解压
    import zipfile
    for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
        if 'webp' in z.lower() or 'image' in z.lower():
            with zipfile.ZipFile(z) as f: f.extractall('/content')
    img_path = glob.glob(f'/content/**/{target_iid}*.webp', recursive=True)
print('图片:', img_path[0] if img_path else '未找到')

# 3) 用模型预测这张图,拿预测的关键点
res = model.predict(img_path[0], imgsz=960, conf=0.5, verbose=False)[0]

# 模型输出的关键点:res.keypoints.data shape (人数, 24, 3) = [x, y, conf]
kpts = res.keypoints.data[0].cpu().numpy()  # 取第一个人(主滑手)
W, H = res.orig_shape[1], res.orig_shape[0]

# 把预测点转成 extract_all 需要的格式 [x_norm, y_norm, v]
# 标注是归一化坐标,所以预测点也归一化;v用置信度阈值转(>0.5算可见)
pred_ann = []
for x, y, c in kpts:
    v = 1 if c >= 0.5 else 0
    pred_ann.append([x / W, y / H, v])

# 4) 两种点分别算物理量
gt_result   = extract_all(gt_ann)
pred_result = extract_all(pred_ann)

# 5) 对比
print('\n===== 标注点 vs 预测点 物理量对比 =====')
print(f"{'量':<10}{'标注点':>10}{'预测点':>10}{'差异':>10}")
for name in gt_result['joints']:
    g = gt_result['joints'][name][0]
    p = pred_result['joints'][name][0]
    d = abs(g-p) if not (np.isnan(g) or np.isnan(p)) else float('nan')
    print(f"{name:<10}{g:>9.1f}°{p:>9.1f}°{d:>9.1f}°")
for key, label in [('trunk_tilt','躯干内倾'),('angulation_R','右反弓'),('angulation_L','左反弓')]:
    g, p = gt_result[key][0], pred_result[key][0]
    d = abs(g-p) if not (np.isnan(g) or np.isnan(p)) else float('nan')
    print(f"{label:<10}{g:>9.1f}°{p:>9.1f}°{d:>9.1f}°")
gx, gy = gt_result['com'][0], gt_result['com'][1]
px, py = pred_result['com'][0], pred_result['com'][1]
print(f"{'重心x':<10}{gx:>10.3f}{px:>10.3f}{abs(gx-px):>10.3f}")
print(f"{'重心y':<10}{gy:>10.3f}{py:>10.3f}{abs(gy-py):>10.3f}")

In [ ]:
import json, glob

labels = json.load(open(glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]))

# 统计每个 (video, split) 下有多少帧,找最长的几段
seqs = []
for vid in labels:
    for sid in labels[vid]:
        n = len(labels[vid][sid])
        seqs.append((n, vid, sid))

seqs.sort(reverse=True)
print('最长的10个连续帧序列:')
for n, vid, sid in seqs[:10]:
    print(f'  帧数={n:3d}  video={vid}  split={sid}')

In [ ]:
import json, glob
import numpy as np

labels = json.load(open(glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]))

VID, SID = '5UHRvqx1iuQ', '0'
seq = labels[VID][SID]

# 看这段里的帧 id(键)长什么样,确认能不能按时间排序
iids = list(seq.keys())
print('帧数:', len(iids))
print('前5个帧id:', iids[:5])
print('后5个帧id:', iids[-5:])

In [ ]:
import json, glob, sys
import numpy as np
import matplotlib.pyplot as plt
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics; importlib.reload(ski_physics)
from ski_physics import extract_all

# ---- 1) 取这段63帧,按帧号排序 ----
labels = json.load(open(glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]))
VID, SID = '5UHRvqx1iuQ', '0'
seq = labels[VID][SID]
iids = sorted(seq.keys())   # 已是连续编号,排序=按时间

# ---- 2) 逐帧提取物理量,存成时间序列 ----
T = len(iids)
trunk_tilt = np.full(T, np.nan)   # 躯干内倾(t)
knee_r     = np.full(T, np.nan)   # 右膝角(t)
angu_r     = np.full(T, np.nan)   # 右反弓(t)
com_x      = np.full(T, np.nan)   # 重心x(t)
com_y      = np.full(T, np.nan)   # 重心y(t)

for i, iid in enumerate(iids):
    ann = seq[iid]['annotation']
    r = extract_all(ann)
    # 只在"可信"时记录,不可信留 nan(后面插值)
    if r['trunk_tilt'][1]:   trunk_tilt[i] = r['trunk_tilt'][0]
    if r['joints']['右膝'][1]: knee_r[i]   = r['joints']['右膝'][0]
    if r['angulation_R'][1]: angu_r[i]     = r['angulation_R'][0]
    if r['com'][3]:          com_x[i] = r['com'][0]; com_y[i] = r['com'][1]

# ---- 3) 缺失插值 + 平滑 ----
def fill_and_smooth(arr, win=5):
    """线性插值补 nan,再滑动平均平滑"""
    x = np.arange(len(arr))
    good = ~np.isnan(arr)
    if good.sum() < 2:
        return arr
    # 线性插值补空洞
    filled = np.interp(x, x[good], arr[good])
    # 滑动平均平滑(边缘用'same'保持长度)
    kernel = np.ones(win)/win
    smoothed = np.convolve(filled, kernel, mode='same')
    return smoothed

trunk_s = fill_and_smooth(trunk_tilt)
knee_s  = fill_and_smooth(knee_r)
angu_s  = fill_and_smooth(angu_r)
comx_s  = fill_and_smooth(com_x)

# ---- 4) 画曲线 ----
t = np.arange(T)
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].plot(t, trunk_tilt, 'o', alpha=0.3, label='raw', color='steelblue')
axes[0].plot(t, trunk_s, '-', lw=2, label='smoothed', color='darkblue')
axes[0].axhline(0, color='gray', ls='--', alpha=0.5)
axes[0].set_ylabel('Trunk tilt (deg)\n+right / -left')
axes[0].set_title('Trunk inclination over time  (look for wave pattern = turns!)')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(t, knee_r, 'o', alpha=0.3, color='orange')
axes[1].plot(t, knee_s, '-', lw=2, color='darkorange')
axes[1].set_ylabel('Right knee (deg)')
axes[1].set_title('Knee angle over time')
axes[1].grid(alpha=0.3)

axes[2].plot(t, com_x, 'o', alpha=0.3, color='green')
axes[2].plot(t, comx_s, '-', lw=2, color='darkgreen')
axes[2].set_ylabel('CoM x')
axes[2].set_xlabel('Frame (time →)')
axes[2].set_title('Center of mass (horizontal) over time')
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# 打印缺失情况
print(f'躯干内倾: {np.sum(~np.isnan(trunk_tilt))}/{T} 帧可信')
print(f'右膝角:   {np.sum(~np.isnan(knee_r))}/{T} 帧可信')
print(f'右反弓:   {np.sum(~np.isnan(angu_r))}/{T} 帧可信')
print(f'重心:     {np.sum(~np.isnan(com_x))}/{T} 帧可信')

In [ ]:
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt

# 用平滑后的内倾曲线(前面算的 trunk_s)
curve = trunk_s
t = np.arange(len(curve))

# ---- 1) 找波峰(向右倾的顶点)和波谷(向左倾的顶点)----
PROMINENCE = 8    # 突出度门槛(度):峰要比周围凸出至少8度才算数,过滤小抖动
DISTANCE   = 5    # 两峰最小间隔5帧

peaks, _   = find_peaks(curve,  prominence=PROMINENCE, distance=DISTANCE)  # 波峰(右倾顶点)
valleys, _ = find_peaks(-curve, prominence=PROMINENCE, distance=DISTANCE)  # 波谷(左倾顶点)

# ---- 2) 找过零点(换弯时刻)----
zero_crossings = []
for i in range(len(curve)-1):
    if curve[i] * curve[i+1] < 0:   # 符号变化
        zero_crossings.append(i)
zero_crossings = np.array(zero_crossings)

# ---- 3) 打印识别结果 ----
print('===== 过弯结构识别 =====')
print(f'波峰(右倾顶点)帧: {peaks.tolist()}')
print(f'波谷(左倾顶点)帧: {valleys.tolist()}')
print(f'过零点(换弯)帧:   {zero_crossings.tolist()}')
n_turns = len(peaks) + len(valleys)
print(f'\n识别到约 {n_turns} 次转弯')

# ---- 4) 可视化:在内倾曲线上标出所有结构 ----
plt.figure(figsize=(13, 5))
plt.plot(t, curve, '-', lw=2, color='darkblue', label='trunk tilt (smoothed)')
plt.axhline(0, color='gray', ls='--', alpha=0.5)
plt.plot(peaks,   curve[peaks],   'v', ms=14, color='red',    label='apex (right turn)')
plt.plot(valleys, curve[valleys], '^', ms=14, color='green',  label='apex (left turn)')
for zc in zero_crossings:
    plt.axvline(zc, color='orange', ls=':', alpha=0.7)
plt.plot([], [], color='orange', ls=':', label='turn switch (zero-cross)')
plt.xlabel('Frame (time →)'); plt.ylabel('Trunk tilt (deg)')
plt.title('Detected turn structure: apexes (peaks/valleys) + switches (zero-crossings)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# 只装 ultralytics(scipy用Colab自带的,不升级)
!pip install -U ultralytics

# 挂载 Drive
from google.colab import drive
drive.mount('/content/drive')

# 导入物理模块
import sys
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics
importlib.reload(ski_physics)
from ski_physics import extract_all
print('✅ 环境就绪')

In [ ]:
import json, glob, zipfile
import numpy as np

# 找标签JSON,没有就从Drive重解压
labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)
if not labels_path:
    for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
        if 'labels' in z.lower() and 'ski2dpose' in z.lower():
            with zipfile.ZipFile(z) as f: f.extractall('/content')
    labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)
labels = json.load(open(labels_path[0]))
print('标签加载成功')

# 取那段63帧序列
VID, SID = '5UHRvqx1iuQ', '0'
seq = labels[VID][SID]
iids = sorted(seq.keys())
T = len(iids)

# 逐帧算躯干内倾(只这一个量,够Step3用)
trunk_tilt = np.full(T, np.nan)
for i, iid in enumerate(iids):
    ann = seq[iid]['annotation']
    r = extract_all(ann)
    if r['trunk_tilt'][1]:
        trunk_tilt[i] = r['trunk_tilt'][0]

# 缺失插值 + 平滑
def fill_and_smooth(arr, win=5):
    x = np.arange(len(arr))
    good = ~np.isnan(arr)
    if good.sum() < 2: return arr
    filled = np.interp(x, x[good], arr[good])
    return np.convolve(filled, np.ones(win)/win, mode='same')

trunk_s = fill_and_smooth(trunk_tilt)
print(f'✅ 内倾曲线 trunk_s 重建完成,共{T}帧,可信{np.sum(~np.isnan(trunk_tilt))}帧')

In [ ]:
from scipy.signal import find_peaks
import numpy as np
import matplotlib.pyplot as plt

def detect_turns(curve, prominence=5, distance=5,
                 min_apex_amp=8, min_side_amp=8, side=4):
    peaks_raw, _   = find_peaks(curve,  prominence=prominence, distance=distance)
    valleys_raw, _ = find_peaks(-curve, prominence=prominence, distance=distance)
    apexes = [(int(i), +1) for i in peaks_raw] + [(int(i), -1) for i in valleys_raw]
    apexes = [(i, s) for (i, s) in apexes if abs(curve[i]) >= min_apex_amp]
    apexes.sort()

    zero_crossings = []
    for i in range(len(curve) - 1):
        if curve[i] * curve[i+1] < 0:
            left  = curve[max(0, i-side):i+1]
            right = curve[i+1:min(len(curve), i+1+side)]
            if len(left) == 0 or len(right) == 0:
                continue
            lm, rm = np.mean(left), np.mean(right)
            if lm * rm < 0 and abs(lm) >= min_side_amp and abs(rm) >= min_side_amp:
                zc = i if abs(curve[i]) < abs(curve[i+1]) else i+1
                if not zero_crossings or zc - zero_crossings[-1] >= distance:
                    zero_crossings.append(zc)
    return apexes, zero_crossings


curve = trunk_s
apexes, zero_crossings = detect_turns(curve, prominence=5, distance=5,
                                      min_apex_amp=8, min_side_amp=8, side=4)

final_peaks   = [i for i, s in apexes if s == +1]
final_valleys = [i for i, s in apexes if s == -1]

print('===== 过弯结构识别(只用幅度过滤,保留不对称)=====')
print(f'右倾顶点帧: {final_peaks}')
print(f'左倾顶点帧: {final_valleys}')
print(f'换弯点帧:   {zero_crossings}')
print(f'顶点数: {len(apexes)}   换弯数: {len(zero_crossings)}')

t = np.arange(len(curve))
plt.figure(figsize=(13, 5))
plt.plot(t, curve, '-', lw=2, color='darkblue', label='trunk tilt')
plt.axhline(0, color='gray', ls='--', alpha=0.5)
plt.axhline( 8, color='pink', ls=':', alpha=0.6)
plt.axhline(-8, color='pink', ls=':', alpha=0.6, label='±8 threshold')
plt.plot(final_peaks,   curve[final_peaks],   'v', ms=14, color='red',   label='apex right')
plt.plot(final_valleys, curve[final_valleys], '^', ms=14, color='green', label='apex left')
for zc in zero_crossings:
    plt.axvline(zc, color='orange', ls=':', alpha=0.8)
plt.plot([], [], color='orange', ls=':', label='turn switch')
plt.xlabel('Frame'); plt.ylabel('Trunk tilt (deg)')
plt.title('Turn structure (amplitude filtering only)')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.patches import Patch

def label_phases(curve, apexes, flat_thresh=8, apex_window=2):
    """
    给每帧打过弯阶段标签(用原始曲线,不去基线)。
    参数:
      curve       : 内倾曲线(原始 trunk_s,未去基线)
      apexes      : [(帧, 方向)] Step3识别出的顶点
      flat_thresh : 内倾幅度 < 此值,算"过渡/直行"
      apex_window : 顶点前后多少帧算"顶点"阶段(±2,共5帧)
    返回:
      labels : 每帧阶段标签
    """
    T = len(curve)
    labels = np.array(['transition'] * T, dtype=object)
    abs_curve = np.abs(curve)
    apex_frames = sorted([i for i, s in apexes])

    # 1) 顶点阶段:每个顶点前后 apex_window 帧(接受重叠连片,如实反映持续倾斜)
    for af in apex_frames:
        for j in range(max(0, af-apex_window), min(T, af+apex_window+1)):
            labels[j] = 'apex'

    # 2) 入弯/出弯:看内倾幅度在增大还是减小
    for i in range(T):
        if labels[i] == 'apex':
            continue
        if abs_curve[i] < flat_thresh:
            labels[i] = 'transition'       # 接近直立
            continue
        if i > 0:
            labels[i] = 'initiation' if abs_curve[i] > abs_curve[i-1] else 'completion'
    return labels


# ===== 运行 =====
labels = label_phases(curve, apexes, flat_thresh=8, apex_window=2)

# 中文名映射(用于打印)
cn = {'initiation':'入弯', 'apex':'顶点', 'completion':'出弯', 'transition':'过渡'}

# ===== 诚实标注 =====
print('⚠️ 注意:本段内倾整体偏负(中位数 %.1f°),可能因相机倾斜或滑手持续左倾;' % np.median(curve))
print('   且2D视角本身存在透视失真。以下阶段标签据原始曲线得出,未做视角/基线校正。\n')

# ===== 逐帧打印 =====
print('===== 每帧过弯阶段标签 =====')
icon = {'入弯':'▲ 入弯', '顶点':'● 顶点', '出弯':'▼ 出弯', '过渡':'— 过渡'}
for i in range(len(curve)):
    print(f'帧{i:2d}: 内倾{curve[i]:+6.1f}°  {icon[cn[labels[i]]]}')

# ===== 阶段分布 =====
cnt = Counter(labels)
print('\n阶段分布:', {cn[k]: v for k, v in cnt.items()})

# ===== 可视化(英文标签,避免字体乱码)=====
phase_colors = {'initiation':'orange', 'apex':'red', 'completion':'green', 'transition':'lightgray'}
phase_en     = {'initiation':'Initiation', 'apex':'Apex', 'completion':'Completion', 'transition':'Transition'}
t = np.arange(len(curve))

plt.figure(figsize=(14, 5))
plt.plot(t, curve, '-', lw=1.5, color='darkblue', zorder=3)
plt.axhline(0, color='gray', ls='--', alpha=0.5)
for i in range(len(curve)):
    plt.axvspan(i-0.5, i+0.5, color=phase_colors[labels[i]], alpha=0.4, zorder=1)
legend = [Patch(color=c, alpha=0.5, label=phase_en[p]) for p, c in phase_colors.items()]
plt.legend(handles=legend, loc='upper right')
plt.xlabel('Frame (time -->)'); plt.ylabel('Trunk tilt (deg)')
plt.title('Per-frame turn phase labels (raw curve, no baseline correction)')
plt.tight_layout(); plt.show()

In [ ]:
# 1. 重装
!pip install -U ultralytics

# 2. 挂Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. 导入物理模块
import sys
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics; importlib.reload(ski_physics)
from ski_physics import extract_all
print('✅ 就绪,明天继续第五关')

In [ ]:
import json, glob, zipfile, sys
import numpy as np
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics; importlib.reload(ski_physics)
from ski_physics import extract_all

# ---- 1) 找标签,没有就从Drive重解压 ----
labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)
if not labels_path:
    print('临时盘没有标签,从Drive解压...')
    for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
        if 'label' in z.lower() and 'ski2dpose' in z.lower():
            with zipfile.ZipFile(z) as f: f.extractall('/content')
            print('解压:', z)
    labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)

if not labels_path:
    print('❌ 还是没找到标签JSON。请确认Drive里标签zip的名字,可跑:')
    print("   import glob; print(glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True))")
else:
    all_labels = json.load(open(labels_path[0]))
    seq = all_labels['5UHRvqx1iuQ']['0']
    iids = sorted(seq.keys())
    T = len(iids)
    print(f'✅ 标签加载成功,{T}帧')

    # ---- 2) 重建内倾曲线 trunk_s ----
    trunk_tilt = np.full(T, np.nan)
    for i, iid in enumerate(iids):
        r = extract_all(seq[iid]['annotation'])
        if r['trunk_tilt'][1]:
            trunk_tilt[i] = r['trunk_tilt'][0]
    def fill_and_smooth(arr, win=5):
        x = np.arange(len(arr)); good = ~np.isnan(arr)
        if good.sum() < 2: return arr
        filled = np.interp(x, x[good], arr[good])
        return np.convolve(filled, np.ones(win)/win, mode='same')
    trunk_s = fill_and_smooth(trunk_tilt)

    # ---- 3) 重建转弯识别 + 阶段标签(第四关)----
    from scipy.signal import find_peaks
    def detect_turns(curve, prominence=5, distance=5, min_apex_amp=8, min_side_amp=8, side=4):
        pk,_ = find_peaks(curve, prominence=prominence, distance=distance)
        vl,_ = find_peaks(-curve, prominence=prominence, distance=distance)
        apex = [(int(i),+1) for i in pk]+[(int(i),-1) for i in vl]
        apex = [(i,s) for i,s in apex if abs(curve[i])>=min_apex_amp]; apex.sort()
        zc=[]
        for i in range(len(curve)-1):
            if curve[i]*curve[i+1]<0:
                l=curve[max(0,i-side):i+1]; r=curve[i+1:min(len(curve),i+1+side)]
                if len(l)and len(r):
                    lm,rm=np.mean(l),np.mean(r)
                    if lm*rm<0 and abs(lm)>=min_side_amp and abs(rm)>=min_side_amp:
                        z=i if abs(curve[i])<abs(curve[i+1]) else i+1
                        if not zc or z-zc[-1]>=distance: zc.append(z)
        return apex, zc
    apexes, zero_crossings = detect_turns(trunk_s)

    def label_phases(curve, apexes, flat_thresh=8, apex_window=2):
        T=len(curve); labels=np.array(['过渡']*T,dtype=object); ac=np.abs(curve)
        for af,_ in apexes:
            for j in range(max(0,af-apex_window),min(T,af+apex_window+1)): labels[j]='顶点'
        for i in range(T):
            if labels[i]=='顶点': continue
            if ac[i]<flat_thresh: labels[i]='过渡'; continue
            if i>0: labels[i]='入弯' if ac[i]>ac[i-1] else '出弯'
        return labels
    labels = label_phases(trunk_s, apexes)
    print('✅ 阶段标签 labels 重建完成')
    print('   现在可以跑 A-frame 代码了(seq, iids, labels 都已就绪)')

In [ ]:
import numpy as np

# 关键点索引(来自 ski_physics)
HIP_R, KNEE_R, ANK_R = 10, 11, 12
HIP_L, KNEE_L, ANK_L = 13, 14, 15

def a_frame_index(ann, vis_thresh=1, eps=1e-6):
    """
    计算单帧的 A-frame 指标 = 脚间距 / 膝间距(水平方向)。
    >1: 脚比膝开(A-frame倾向);越大越明显。约等于1或<1: 正常。
    返回 (指标, 膝间距, 脚间距, 可信)
    """
    kR, kL = np.asarray(ann[KNEE_R],float), np.asarray(ann[KNEE_L],float)
    aR, aL = np.asarray(ann[ANK_R],float),  np.asarray(ann[ANK_L],float)

    # 四个点都要可见
    ok = all(p[2] >= vis_thresh for p in [kR, kL, aR, aL])
    if not ok:
        return np.nan, np.nan, np.nan, False

    knee_dist = abs(kL[0] - kR[0])   # 膝间距(水平)
    foot_dist = abs(aL[0] - aR[0])   # 脚间距(水平)

    if knee_dist < eps:
        return np.nan, knee_dist, foot_dist, False  # 膝几乎重合,比值无意义

    return foot_dist / knee_dist, knee_dist, foot_dist, True


# ===== 在63帧序列上逐帧算,结合阶段标签 =====
import json, glob
labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]
all_labels = json.load(open(labels_path))
seq = all_labels['5UHRvqx1iuQ']['0']
iids = sorted(seq.keys())

af_index = np.full(len(iids), np.nan)
for i, iid in enumerate(iids):
    ann = seq[iid]['annotation']
    idx, kd, fd, ok = a_frame_index(ann)
    if ok:
        af_index[i] = idx

# 打印,并和阶段标签对照(labels来自第四关,需已运行)
print('===== A-frame 指标(脚间距/膝间距)=====')
print('指标>1.3 提示A-frame倾向(阈值经验值,待校准)\n')
AFRAME_THRESH = 1.3   # 经验阈值:脚间距超过膝间距30%以上算明显
for i, iid in enumerate(iids):
    if np.isnan(af_index[i]):
        continue
    phase = labels[i] if 'labels' in dir() else '?'   # 第四关的每帧阶段
    flag = '  ⚠️ A-frame' if af_index[i] > AFRAME_THRESH else ''
    print(f'帧{i:2d} [{phase:11s}] A-frame指标={af_index[i]:.2f}{flag}')

# 统计
valid = af_index[~np.isnan(af_index)]
print(f'\n有效帧: {len(valid)}/{len(iids)}')
print(f'A-frame指标 中位数={np.median(valid):.2f}  范围=[{np.min(valid):.2f}, {np.max(valid):.2f}]')
print(f'超过阈值{AFRAME_THRESH}的帧数: {np.sum(valid > AFRAME_THRESH)}')

In [ ]:
import numpy as np

# 平滑 A-frame 指标(消除单帧尖峰)
def smooth_nan(arr, win=5):
    x = np.arange(len(arr)); good = ~np.isnan(arr)
    if good.sum() < 2: return arr
    filled = np.interp(x, x[good], arr[good])
    return np.convolve(filled, np.ones(win)/win, mode='same')

af_smooth = smooth_nan(af_index)

# 按阶段聚合:看每个阶段的平均 A-frame(只在可信帧上)
from collections import defaultdict
phase_af = defaultdict(list)
for i in range(len(iids)):
    if not np.isnan(af_index[i]):
        phase_af[labels[i]].append(af_index[i])

print('===== 各阶段 A-frame 指标(聚合,看持续性)=====')
for ph in ['入弯','顶点','出弯','过渡']:
    if phase_af[ph]:
        vals = np.array(phase_af[ph])
        print(f'{ph:4s}: 平均={vals.mean():.2f}  中位数={np.median(vals):.2f}  '
              f'帧数={len(vals)}  最大={vals.max():.2f}')

# 平滑后再看超阈值情况
valid_s = af_smooth[~np.isnan(af_smooth)]
print(f'\n平滑后: 中位数={np.median(valid_s):.2f}  '
      f'超阈值1.3的帧={np.sum(valid_s>1.3)}/{len(valid_s)}')
print('(平滑后超阈值帧数若大幅减少,说明原来的尖峰多是噪声)')

In [ ]:
import numpy as np

af_init = np.mean(phase_af['入弯']) if phase_af['入弯'] else np.nan
af_comp = np.mean(phase_af['出弯']) if phase_af['出弯'] else np.nan
af_apex = np.mean(phase_af['顶点']) if phase_af['顶点'] else np.nan

print('===== A-frame 规则判断(阶段对比)=====')
print(f'入弯平均: {af_init:.2f}   出弯平均: {af_comp:.2f}   顶点平均: {af_apex:.2f}')

# 规则:入弯明显高于出弯(差值>0.15)且入弯本身偏高(>1.2)→ 提示
if not np.isnan(af_init) and not np.isnan(af_comp):
    diff = af_init - af_comp
    if af_init > 1.2 and diff > 0.15:
        print(f'\n💬 建议:检测到入弯阶段脚比膝明显偏开(指标{af_init:.2f},'
              f'比出弯高{diff:.2f}),可能存在轻度A-frame倾向。')
        print('   这通常提示入弯时内侧腿不够主动内倾。可尝试:入弯时')
        print('   主动用内侧脚踝引导倾倒,让内侧膝更早跟上外侧腿。')
        print('   ⚠️ 注:斜侧视角下此指标有误差,且为轻度,仅供参考。')
    else:
        print('\n✅ 未检测到明显的阶段性A-frame倾向。')

In [ ]:
import numpy as np
from collections import defaultdict

KNEE_R, KNEE_L = 11, 14
# joints里的键名(来自ski_physics的all_joint_angles)
# '右膝','左膝'

def inner_outer_knee_check(seq, iids, labels, trunk_s, apex_window=2):
    """
    规则2:顶点附近比较内侧膝角 vs 外侧膝角。
    内侧腿应屈得更深(膝角更小)。若内侧≥外侧,提示内侧腿偷懒。
    A策略:假设正后方拍,内倾负=左弯=左腿内侧。
    """
    from ski_physics import extract_all

    # 找出顶点帧(labels=='顶点'的连续段的中心,简化为所有顶点帧)
    apex_frames = [i for i in range(len(iids)) if labels[i]=='顶点']

    results = []   # 每个顶点帧的 (帧, 弯向, 内侧膝角, 外侧膝角)
    for i in apex_frames:
        ann = seq[iids[i]]['annotation']
        r = extract_all(ann)
        kr, ok_r = r['joints']['右膝']
        kl, ok_l = r['joints']['左膝']
        if not (ok_r and ok_l):     # 两膝角都要可信
            continue
        tilt = trunk_s[i]
        if tilt < 0:                # 左弯:内侧=左腿
            inner, outer, side = kl, kr, '左弯(内侧=左腿)'
        else:                       # 右弯:内侧=右腿
            inner, outer, side = kr, kl, '右弯(内侧=右腿)'
        results.append((i, side, inner, outer))

    return results


# ===== 运行 =====
results = inner_outer_knee_check(seq, iids, labels, trunk_s)

print('===== 规则2:内侧腿屈曲检查(顶点附近)=====')
print('准则:内侧膝角应 < 外侧膝角(内侧腿屈得更深)\n')
print(f"{'帧':>3} {'弯向':<16} {'内侧膝':>7} {'外侧膝':>7} {'差(内-外)':>9}  判断")

inner_lazy_count = 0
diffs = []
for i, side, inner, outer in results:
    diff = inner - outer
    diffs.append(diff)
    # 内侧膝角 >= 外侧(差>=0)说明内侧没屈更深 → 偷懒
    judge = '⚠️ 内侧偏直' if diff >= 0 else 'OK(内侧更屈)'
    if diff >= 0:
        inner_lazy_count += 1
    print(f"{i:>3} {side:<16} {inner:>6.1f}° {outer:>6.1f}° {diff:>+8.1f}°  {judge}")

# ===== 聚合判断 =====
print(f'\n----- 聚合 -----')
if diffs:
    mean_diff = np.mean(diffs)
    print(f'内侧-外侧膝角 平均差: {mean_diff:+.1f}°  (负=内侧更屈,健康;正=内侧偏直,问题)')
    print(f'内侧偏直的顶点数: {inner_lazy_count}/{len(diffs)}')
    if mean_diff >= -3:   # 内侧没明显更屈(差>-3度)→ 提示
        print(f'\n💬 建议:顶点附近内侧腿屈曲不足(平均仅比外侧{"深" if mean_diff<0 else "还浅"}'
              f'{abs(mean_diff):.0f}°)。内侧腿应主动深屈、收短,'
              f'为身体内倾让出空间,帮助建立更大刃角。')
        print('   ⚠️ 注:基于"正后方拍摄"假设判断内外侧;斜侧视角膝角有误差,仅供参考。')
    else:
        print(f'\n✅ 内侧腿屈曲合理(平均比外侧深{abs(mean_diff):.0f}°)。')
else:
    print('顶点附近无足够可信的双膝角数据。')

In [ ]:
import glob
from IPython.display import Image, display

# 找这段的图片(之前验证时见过路径:/content/Images_webp/5UHRvqx1iuQ/0/...)
imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))
print(f'找到 {len(imgs)} 张图')
if imgs:
    # 看第10帧和第40帧(那个异常帧)
    for idx in [10, 40]:
        p = [x for x in imgs if f'_{idx:05d}' in x]
        if p:
            print(f'\n帧{idx}:')
            display(Image(p[0]))

In [ ]:
import glob
# 看Drive里所有zip,找图片包
zips = glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True)
for z in zips:
    print(z)

### Why is every turn classified as "right"?

The turn detection kept labeling almost every apex as the same direction, and it is impossible for a real skiing run. I load the raw video
frames to see what is actually happening and realize that the footage was shot from a distant, front-facing, downward angle. In that viewpoint, the left–right inclination is compressed by perspective and its sign becomes unreliable.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))
print(f'找到 {len(imgs)} 张图')

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, idx in zip(axes, [10, 40]):
    p = [x for x in imgs if f'_{idx:05d}' in x]
    if p:
        img = Image.open(p[0])
        ax.imshow(img)
        ax.set_title(f'Frame {idx}')
        ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# 前拍修正:内倾负(图像左倾)= 滑手向右倾 = 右弯 = 内侧右腿
# 内倾正(图像右倾)= 滑手向左倾 = 左弯 = 内侧左腿
# (与之前A策略假设相反)

results_fixed = []
for i in [x for x in range(len(iids)) if labels[i]=='顶点']:
    from ski_physics import extract_all
    r = extract_all(seq[iids[i]]['annotation'])
    kr, ok_r = r['joints']['右膝']
    kl, ok_l = r['joints']['左膝']
    if not (ok_r and ok_l):
        continue
    # 数值合理性过滤(剔除极端值,如帧40)
    if not (90 <= kr <= 180 and 90 <= kl <= 180):
        continue
    tilt = trunk_s[i]
    if tilt < 0:                # 前拍:图像左倾=右弯=内侧右腿
        inner, outer, side = kr, kl, '右弯(内侧=右腿)'
    else:                       # 图像右倾=左弯=内侧左腿
        inner, outer, side = kl, kr, '左弯(内侧=左腿)'
    results_fixed.append((i, side, inner, outer))

print('===== 规则2 修正版(前拍 + 过滤极端值)=====')
print(f"{'帧':>3} {'弯向':<16} {'内侧膝':>7} {'外侧膝':>7} {'差':>7}")
diffs = []
for i, side, inner, outer in results_fixed:
    d = inner - outer
    diffs.append(d)
    print(f"{i:>3} {side:<16} {inner:>6.1f}° {outer:>6.1f}° {d:>+6.1f}°")

import numpy as np
if diffs:
    print(f'\n保留 {len(diffs)} 帧(已过滤遮挡+极端值)')
    print(f'内侧-外侧膝角差: 中位数={np.median(diffs):+.1f}°  平均={np.mean(diffs):+.1f}°')
    med = np.median(diffs)
    if med >= -3:
        print(f'\n💬 提示:内侧腿屈曲偏少(中位数{med:+.0f}°),仅供参考。')
    else:
        print(f'\n✅ 内侧腿屈曲合理(中位数比外侧深{abs(med):.0f}°)——健康。')
    print('   (已修正为正前方拍摄;斜侧/视角仍有误差)')

In [ ]:
import json, glob, sys
import numpy as np
from scipy.signal import find_peaks
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics; importlib.reload(ski_physics)
from ski_physics import extract_all

# ---- 重建数据 ----
labels_path = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)[0]
all_labels = json.load(open(labels_path))
seq = all_labels['5UHRvqx1iuQ']['0']
iids = sorted(seq.keys())
T = len(iids)

# 内倾曲线
trunk_tilt = np.full(T, np.nan)
for i, iid in enumerate(iids):
    r = extract_all(seq[iid]['annotation'])
    if r['trunk_tilt'][1]:
        trunk_tilt[i] = r['trunk_tilt'][0]
def fill_and_smooth(arr, win=5):
    x=np.arange(len(arr)); good=~np.isnan(arr)
    if good.sum()<2: return arr
    return np.convolve(np.interp(x,x[good],arr[good]), np.ones(win)/win, mode='same')
trunk_s = fill_and_smooth(trunk_tilt)

# 转弯识别
def detect_turns(curve, prominence=5, distance=5, min_apex_amp=8, min_side_amp=8, side=4):
    pk,_=find_peaks(curve,prominence=prominence,distance=distance)
    vl,_=find_peaks(-curve,prominence=prominence,distance=distance)
    ap=[(int(i),+1) for i in pk]+[(int(i),-1) for i in vl]
    ap=[(i,s) for i,s in ap if abs(curve[i])>=min_apex_amp]; ap.sort()
    return ap
apexes = detect_turns(trunk_s)

# 阶段标签
def label_phases(curve, apexes, flat_thresh=8, apex_window=2):
    T=len(curve); lab=np.array(['过渡']*T,dtype=object); ac=np.abs(curve)
    for af,_ in apexes:
        for j in range(max(0,af-apex_window),min(T,af+apex_window+1)): lab[j]='顶点'
    for i in range(T):
        if lab[i]=='顶点': continue
        if ac[i]<flat_thresh: lab[i]='过渡'; continue
        if i>0: lab[i]='入弯' if ac[i]>ac[i-1] else '出弯'
    return lab
labels = label_phases(trunk_s, apexes)

# ---- 验证状态正确 ----
from collections import Counter
print('✅ 重建完成,验证状态:')
print('阶段分布:', dict(Counter(labels)))
apex_frames = [i for i in range(T) if labels[i]=='顶点']
print(f'顶点帧数: {len(apex_frames)}  → {apex_frames}')
# 看这些顶点帧的内倾符号(应该有正有负)
signs = [('右倾+' if trunk_s[i]>0 else '左倾-') for i in apex_frames]
print(f'顶点帧内倾符号: {signs}')

In [ ]:
import glob, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))

# 看几个关键帧:不同内倾值的帧,判断滑手实际倾斜方向
check_frames = [11, 20, 25, 40, 59]   # 谷、峰附近、谷、谷
fig, axes = plt.subplots(1, len(check_frames), figsize=(20, 5))
for ax, idx in zip(axes, check_frames):
    p = [x for x in imgs if f'_{idx:05d}' in x]
    if p:
        ax.imshow(Image.open(p[0]))
        ax.set_title(f'Frame {idx}\ntilt={trunk_s[idx]:+.1f}', fontsize=11)
        ax.axis('off')
plt.tight_layout(); plt.show()

print('看这几帧,判断滑手身体实际倒向哪边:')
print('如果滑手在这些帧里【明显左右交替倾斜】→ 偏负是相机/视角造成的假象')
print('如果滑手【一直向同一边倾(他的右)】→ 偏负是真实的,这段主要在右弯')

### Building a data quality gate

Since camera viewpoint can silently corrupt the inclination signal, I decided that the system should detect this and lower its own confidence, rather than giving confident but wrong advice. This gate checks occlusion rates, whether all detected turns fall in one direction (a sign of viewpoint distortion), and baseline drift in the inclination signal.

In [ ]:
import numpy as np

def assess_data_quality(seq, iids, trunk_s, apexes, labels):
    """
    数据质量门控:分析前先判断这段数据配不配做内倾相关分析。
    返回 dict:各项质量指标 + 总体置信度等级 + 警示。
    """
    from ski_physics import extract_all
    T = len(iids)
    report = {'warnings': [], 'metrics': {}}

    # ---- 检查1:各物理量的遮挡/可信率 ----
    trust_count = {'内倾':0, '重心':0, '膝角':0, '反弓':0}
    for iid in iids:
        r = extract_all(seq[iid]['annotation'])
        if r['trunk_tilt'][1]: trust_count['内倾'] += 1
        if r['com'][3]:        trust_count['重心'] += 1
        if r['joints']['右膝'][1] and r['joints']['左膝'][1]: trust_count['膝角'] += 1
        if r['angulation_R'][1]: trust_count['反弓'] += 1
    for k, v in trust_count.items():
        rate = v / T
        report['metrics'][f'{k}可信率'] = rate
        if rate < 0.5:
            report['warnings'].append(f'⚠️ {k}可信率仅{rate:.0%},遮挡严重,相关建议不可靠')

    # ---- 检查2:视角合理性(今天的核心教训)----
    # 识别出的顶点,如果几乎全是同一方向,极可能是视角失真而非真实
    apex_signs = [np.sign(trunk_s[i]) for i, _ in apexes]
    if apex_signs:
        pos = sum(1 for s in apex_signs if s > 0)
        neg = sum(1 for s in apex_signs if s < 0)
        total = len(apex_signs)
        minority_ratio = min(pos, neg) / total if total > 0 else 0
        report['metrics']['左右弯平衡度'] = minority_ratio
        # 修正检查2的措辞:不断言,而是提示+列出可能性
        if minority_ratio < 0.2:
          report['warnings'].append(
            f'⚠️ 识别出的转弯几乎全是同一方向({max(pos,neg)}/{total})。'
            f'这可能是:(a)视角问题——正对/俯视镜头使内倾方向测量失真;'
            f'或(b)真实情况——这段确实以同方向弯为主。'
            f'两者无法仅凭数据区分。建议:确认拍摄角度,或换一段侧后方、'
            f'相机水平的视频。在确认前,内倾【方向】相关分析置信度低,'
            f'但内倾【幅度】(倾斜程度)仍可参考。')
          report['viewpoint_ok'] = False
        else:
            report['viewpoint_ok'] = True

    # ---- 检查3:内倾基线偏移 ----
    median_tilt = np.median(trunk_s)
    report['metrics']['内倾中位数'] = median_tilt
    if abs(median_tilt) > 8:
        report['warnings'].append(
            f'⚠️ 内倾中位数{median_tilt:+.1f}°明显偏离0,'
            f'存在基线偏移(相机倾斜或视角问题),内倾绝对方向需谨慎。')

    # ---- 总体置信度 ----
    if not report.get('viewpoint_ok', True):
        report['overall'] = '低'
    elif len(report['warnings']) >= 2:
        report['overall'] = '中'
    else:
        report['overall'] = '高'

    return report


# ===== 运行:对当前数据做质量评估 =====
quality = assess_data_quality(seq, iids, trunk_s, apexes, labels)

print('='*50)
print('数据质量门控报告')
print('='*50)
print('\n【质量指标】')
for k, v in quality['metrics'].items():
    if '率' in k or '平衡' in k:
        print(f'  {k}: {v:.0%}')
    else:
        print(f'  {k}: {v:+.1f}')
print('\n【警示】')
if quality['warnings']:
    for w in quality['warnings']:
        print(f'  {w}')
else:
    print('  无明显数据质量问题')
print(f'\n【总体数据置信度】: {quality["overall"]}')

In [ ]:
import numpy as np
from collections import defaultdict
from ski_physics import extract_all

# ========== 规则1:A-frame(依赖:膝踝距离 + 内倾方向分阶段)==========
def rule_aframe(seq, iids, labels, trunk_s):
    KNEE_R, KNEE_L, ANK_R, ANK_L = 11, 14, 12, 15
    phase_af = defaultdict(list)
    for i, iid in enumerate(iids):
        ann = seq[iid]['annotation']
        kR,kL,aR,aL = [np.asarray(ann[x],float) for x in [KNEE_R,KNEE_L,ANK_R,ANK_L]]
        if any(p[2]<1 for p in [kR,kL,aR,aL]): continue
        kd = abs(kL[0]-kR[0]); fd = abs(aL[0]-aR[0])
        if kd < 1e-6: continue
        phase_af[labels[i]].append(fd/kd)
    init = np.mean(phase_af['入弯']) if phase_af['入弯'] else np.nan
    comp = np.mean(phase_af['出弯']) if phase_af['出弯'] else np.nan
    if np.isnan(init) or np.isnan(comp):
        return {'触发':False, '建议':'', '证据':'数据不足', '依赖':['膝踝距离'], '规则置信度':'中'}
    diff = init - comp
    triggered = (init > 1.2 and diff > 0.15)
    return {
        '触发': triggered,
        '建议': '入弯时脚比膝明显偏开,可能存在轻度A-frame倾向,提示内侧腿不够主动内倾。'
                '可尝试入弯时用内侧脚踝引导倾倒,让内侧膝更早跟上。' if triggered else '',
        '证据': f'入弯A-frame指标{init:.2f} vs 出弯{comp:.2f}(差{diff:+.2f})',
        '依赖': ['膝踝距离'],   # 注意:不依赖内倾"方向",只用阶段
        '规则置信度': '中',     # 距离比值较稳健,但斜侧视角有残留误差
    }

# ========== 规则2:内侧腿屈曲(依赖:膝角 + 内倾方向定左右弯)==========
def rule_inner_leg(seq, iids, labels, trunk_s, front_view=True):
    diffs = []
    for i in range(len(iids)):
        if labels[i] != '顶点': continue
        r = extract_all(seq[iids[i]]['annotation'])
        kr, ok_r = r['joints']['右膝']; kl, ok_l = r['joints']['左膝']
        if not (ok_r and ok_l): continue
        if not (90<=kr<=180 and 90<=kl<=180): continue   # 过滤极端值
        tilt = trunk_s[i]
        # front_view: 图像左倾=滑手右倾=右弯=内侧右腿
        if (tilt < 0) == front_view:
            inner, outer = kr, kl
        else:
            inner, outer = kl, kr
        diffs.append(inner - outer)
    if not diffs:
        return {'触发':False, '建议':'', '证据':'数据不足', '依赖':['膝角','内倾方向'], '规则置信度':'低'}
    med = np.median(diffs)
    triggered = (med >= -3)
    return {
        '触发': triggered,
        '建议': '顶点附近内侧腿屈曲偏少,内侧腿应主动深屈收短,为身体内倾让空间、建立更大刃角。'
                if triggered else '',
        '证据': f'内侧-外侧膝角中位数{med:+.0f}°(共{len(diffs)}个顶点)',
        '依赖': ['膝角', '内倾方向'],   # 依赖内倾方向判左右弯!
        '规则置信度': '低',   # 膝角对比噪声大 + 依赖前后镜像假设
    }

# ========== 置信度传导 ==========
def combine_confidence(rule_conf, dependencies, quality):
    """规则本身置信度 + 数据对所依赖量的置信度,取最弱"""
    level = {'高':3, '中':2, '低':1}
    final = level[rule_conf]
    notes = []
    # 内倾方向不可信 → 拖累依赖它的规则
    if '内倾方向' in dependencies and not quality.get('viewpoint_ok', True):
        final = min(final, 1)
        notes.append('数据视角问题使内倾方向不可信')
    # 各量可信率低 → 拖累
    dep_metric = {'膝角':'膝角可信率','膝踝距离':'膝角可信率','重心':'重心可信率','反弓':'反弓可信率'}
    for d in dependencies:
        if d in dep_metric:
            rate = quality['metrics'].get(dep_metric[d], 1.0)
            if rate < 0.5: final = min(final, 1); notes.append(f'{d}遮挡严重')
    return {3:'高',2:'中',1:'低'}[final], notes

# ========== 主引擎 ==========
def run_expert_system(seq, iids, labels, trunk_s, quality):
    rules = [
        ('A-frame检测', rule_aframe(seq, iids, labels, trunk_s)),
        ('内侧腿屈曲', rule_inner_leg(seq, iids, labels, trunk_s)),
    ]
    print('='*55)
    print('滑雪技术分析报告')
    print('='*55)
    print(f'\n【数据置信度】总体: {quality["overall"]}')
    if quality['overall'] == '低':
        print('⚠️ 本段数据质量受限,以下所有建议请谨慎参考。')
    print('\n【技术建议】')
    any_trigger = False
    for name, res in rules:
        final_conf, notes = combine_confidence(res['规则置信度'], res['依赖'], quality)
        if res['触发']:
            any_trigger = True
            print(f'\n● {name}  [置信度: {final_conf}]')
            print(f'  {res["建议"]}')
            print(f'  依据: {res["证据"]}')
            if notes:
                print(f'  ⚠️ 置信度受限: {"; ".join(notes)}')
        else:
            print(f'\n○ {name}: 未触发或数据不足({res["证据"]})')
    if not any_trigger:
        print('\n  未检测到明显技术问题(或数据不足以判断)。')

# ===== 运行 =====
run_expert_system(seq, iids, labels, trunk_s, quality)

In [ ]:
# 只装 ultralytics(scipy用自带的,别升级,否则弹重启)
!pip install -U ultralytics

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics; importlib.reload(ski_physics)
from ski_physics import extract_all
print('✅ 第1段完成')

In [ ]:
import json, glob, zipfile
import numpy as np
from scipy.signal import find_peaks
from collections import Counter, defaultdict

# ---- 标签(没有就从Drive解压)----
lp = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)
if not lp:
    for z in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True):
        if 'label' in z.lower() and 'ski2dpose' in z.lower():
            with zipfile.ZipFile(z) as f: f.extractall('/content')
    lp = glob.glob('/content/**/ski2dpose_labels.json', recursive=True)
all_labels = json.load(open(lp[0]))
seq = all_labels['5UHRvqx1iuQ']['0']
iids = sorted(seq.keys()); T = len(iids)

# ---- 内倾曲线 ----
trunk_tilt = np.full(T, np.nan)
for i, iid in enumerate(iids):
    r = extract_all(seq[iid]['annotation'])
    if r['trunk_tilt'][1]: trunk_tilt[i] = r['trunk_tilt'][0]
def fill_and_smooth(a, win=5):
    x=np.arange(len(a)); g=~np.isnan(a)
    if g.sum()<2: return a
    return np.convolve(np.interp(x,x[g],a[g]), np.ones(win)/win, mode='same')
trunk_s = fill_and_smooth(trunk_tilt)

# ---- 转弯识别 ----
def detect_turns(curve, prominence=5, distance=5, min_apex_amp=8, min_side_amp=8, side=4):
    pk,_=find_peaks(curve,prominence=prominence,distance=distance)
    vl,_=find_peaks(-curve,prominence=prominence,distance=distance)
    ap=[(int(i),+1) for i in pk]+[(int(i),-1) for i in vl]
    ap=[(i,s) for i,s in ap if abs(curve[i])>=min_apex_amp]; ap.sort()
    return ap
apexes = detect_turns(trunk_s)

# ---- 阶段标签 ----
def label_phases(curve, apexes, flat_thresh=8, apex_window=2):
    T=len(curve); lab=np.array(['过渡']*T,dtype=object); ac=np.abs(curve)
    for af,_ in apexes:
        for j in range(max(0,af-apex_window),min(T,af+apex_window+1)): lab[j]='顶点'
    for i in range(T):
        if lab[i]=='顶点': continue
        if ac[i]<flat_thresh: lab[i]='过渡'; continue
        if i>0: lab[i]='入弯' if ac[i]>ac[i-1] else '出弯'
    return lab
labels = label_phases(trunk_s, apexes)

# ---- 数据质量门控 ----
def assess_data_quality(seq, iids, trunk_s, apexes, labels):
    T=len(iids); report={'warnings':[], 'metrics':{}}
    tc={'内倾':0,'重心':0,'膝角':0,'反弓':0}
    for iid in iids:
        r=extract_all(seq[iid]['annotation'])
        if r['trunk_tilt'][1]: tc['内倾']+=1
        if r['com'][3]: tc['重心']+=1
        if r['joints']['右膝'][1] and r['joints']['左膝'][1]: tc['膝角']+=1
        if r['angulation_R'][1]: tc['反弓']+=1
    for k,v in tc.items():
        rate=v/T; report['metrics'][f'{k}可信率']=rate
        if rate<0.5: report['warnings'].append(f'⚠️ {k}可信率仅{rate:.0%},遮挡严重')
    signs=[np.sign(trunk_s[i]) for i,_ in apexes]
    if signs:
        pos=sum(1 for s in signs if s>0); neg=sum(1 for s in signs if s<0)
        tot=len(signs); mr=min(pos,neg)/tot if tot else 0
        report['metrics']['左右弯平衡度']=mr
        if mr<0.2:
            report['warnings'].append(f'⚠️ 转弯几乎全同方向({max(pos,neg)}/{tot}),可能视角失真或真同向弯,内倾方向不可信')
            report['viewpoint_ok']=False
        else: report['viewpoint_ok']=True
    mt=np.median(trunk_s); report['metrics']['内倾中位数']=mt
    if abs(mt)>8: report['warnings'].append(f'⚠️ 内倾中位数{mt:+.1f}°偏离0,基线偏移')
    report['overall']='低' if not report.get('viewpoint_ok',True) else ('中' if len(report['warnings'])>=2 else '高')
    return report
quality = assess_data_quality(seq, iids, trunk_s, apexes, labels)

print('✅ 第2段完成,状态恢复:')
print('  阶段分布:', dict(Counter(labels)))
print('  数据置信度:', quality['overall'])

In [ ]:
import numpy as np

def rule_angulation(seq, iids, labels, quality):
    """
    规则5:外侧腿反弓充分性。外侧腿承压立刃,反弓更大,取左右较大者≈外侧腿。
    只用反弓大小,不依赖内倾方向。
    """
    from ski_physics import extract_all
    ANGU_MIN = 10
    apex_angu = []
    for i in range(len(iids)):
        if labels[i] != '顶点': continue
        r = extract_all(seq[iids[i]]['annotation'])
        vals = []
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals = [v for v in vals if v <= 60]   # 异常过滤:踢掉关键点飞的垃圾值
        if not vals: continue
        apex_angu.append(max(vals))   # 取较大者 = 外侧腿反弓

    if not apex_angu:
        return {'触发':False,'建议':'','证据':'顶点无可信反弓','依赖':['反弓'],'规则置信度':'中'}
    med = np.median(apex_angu)
    triggered = (med < ANGU_MIN)
    return {
        '触发': triggered,
        '建议': '顶点附近外侧腿反弓偏小,可能靠整个身体倒向弯内而非上下身分离来过弯。'
                '可尝试上身更直立、下肢独立倒向弯内,在髋部形成折角以增强立刃抓地。' if triggered else '',
        '证据': f'顶点外侧腿反弓中位数{med:.0f}°(取左右较大者,共{len(apex_angu)}顶点,阈值{ANGU_MIN}°)',
        '依赖': ['反弓'],
        '规则置信度': '中',
    }

res5 = rule_angulation(seq, iids, labels, quality)
print(f"规则5 证据: {res5['证据']}   触发: {res5['触发']}")

In [ ]:
res5 = rule_angulation(seq, iids, labels, quality)
print('===== 规则5:反弓充分性(弱版本+过滤)=====')
print(f'触发: {res5["触发"]}')
print(f'证据: {res5["证据"]}')
print(f'备注: {res5.get("备注","")}')
print(f'规则置信度: {res5["规则置信度"]}')
if res5['触发']:
    print(f'\n建议: {res5["建议"]}')
else:
    print('\n✅ 反弓充分,未触发。')

In [ ]:
import numpy as np

def rule_stance_width(seq, iids, labels, quality):
    """
    规则6:站姿宽度。只在过渡阶段测(中性时刻),避免误判过弯的正常拉开。
    脚间距/髋宽。只用水平距离比值,不依赖内倾方向。
    """
    from ski_physics import extract_all
    HIP_R, HIP_L, ANK_R, ANK_L = 10, 13, 12, 15
    STANCE_MAX = 1.4   # 经验阈值:脚间距>髋宽1.4倍算偏宽(待校准)

    ratios = []
    for i in range(len(iids)):
        if labels[i] != '过渡':      # 只在过渡阶段测
            continue
        ann = seq[iids[i]]['annotation']
        hR,hL,aR,aL = [np.asarray(ann[x],float) for x in [HIP_R,HIP_L,ANK_R,ANK_L]]
        if any(p[2]<1 for p in [hR,hL,aR,aL]):
            continue
        hip_w = abs(hL[0]-hR[0])
        foot_w = abs(aL[0]-aR[0])
        if hip_w < 1e-6:
            continue
        ratio = foot_w / hip_w
        if ratio > 5:               # 异常值过滤(关键点飞了)
            continue
        ratios.append(ratio)

    if not ratios:
        return {'触发':False,'建议':'','证据':'过渡阶段无可信站姿数据',
                '依赖':['脚髋距离'],'规则置信度':'中'}

    med = np.median(ratios)
    triggered = (med > STANCE_MAX)
    return {
        '触发': triggered,
        '建议': f'过渡阶段站姿偏宽(脚间距约为髋宽的{med:.1f}倍)。过宽的站姿会限制'
                f'倾倒和换刃的灵活性。可尝试收窄到与髋同宽,让双脚更靠近、'
                f'用垂直分离(高低差)而非水平张开来过弯。' if triggered else '',
        '证据': f'过渡阶段 脚间距/髋宽 中位数{med:.2f}(共{len(ratios)}帧,阈值{STANCE_MAX})',
        '依赖': ['脚髋距离'],       # 不依赖内倾方向
        '规则置信度': '中',
    }


# ===== 测试 =====
res6 = rule_stance_width(seq, iids, labels, quality)
print('===== 规则6:站姿宽度 =====')
print(f'触发: {res6["触发"]}')
print(f'证据: {res6["证据"]}')
print(f'规则置信度: {res6["规则置信度"]}   依赖: {res6["依赖"]}')
if res6['触发']:
    print(f'\n建议: {res6["建议"]}')
else:
    print('\n✅ 站姿宽度正常,未触发。')

# 顺便看分布
ratios_list = []
from ski_physics import extract_all
for i in range(len(iids)):
    if labels[i]!='过渡': continue
    ann=seq[iids[i]]['annotation']
    hR,hL,aR,aL=[np.asarray(ann[x],float) for x in [10,13,12,15]]
    if any(p[2]<1 for p in [hR,hL,aR,aL]): continue
    hw=abs(hL[0]-hR[0]); fw=abs(aL[0]-aR[0])
    if hw>1e-6 and fw/hw<=5: ratios_list.append(fw/hw)
if ratios_list:
    a=np.array(ratios_list)
    print(f'\n过渡站姿比分布: 中位数={np.median(a):.2f} 范围=[{a.min():.2f},{a.max():.2f}] 共{len(a)}帧')

In [ ]:
import numpy as np

def rule_fore_aft(seq, iids, labels, trunk_s, quality):
    """
    规则3:前后重心分阶段。简化版:看重心相对双脚中点的水平偏移在各阶段的变化。
    权威:出弯靠后正常,不报警;关注的是重心是否在流动(前后循环)。
    诚实:前后重心是矢状面量,2D+视角下测不准,置信度低。
    """
    from ski_physics import extract_all
    ANK_R, ANK_L = 12, 15

    phase_offset = {'入弯':[], '顶点':[], '出弯':[]}
    for i in range(len(iids)):
        ph = labels[i]
        if ph not in phase_offset: continue
        r = extract_all(seq[iids[i]]['annotation'])
        cx, cy, mcov, ok = r['com']
        if not ok: continue
        ann = seq[iids[i]]['annotation']
        aR, aL = np.asarray(ann[ANK_R],float), np.asarray(ann[ANK_L],float)
        if aR[2]<1 or aL[2]<1: continue
        foot_mid_x = (aR[0]+aL[0])/2
        # 重心相对双脚中点的水平偏移(正=重心在脚中点右侧,负=左侧)
        offset = cx - foot_mid_x
        phase_offset[ph].append(offset)

    # 看各阶段的平均偏移,以及入弯vs出弯是否有前后转移
    means = {}
    for ph in ['入弯','顶点','出弯']:
        means[ph] = np.median(phase_offset[ph]) if phase_offset[ph] else np.nan

    # 简化判断:入弯和出弯的重心位置是否有明显差异(重心在流动=健康)
    if np.isnan(means['入弯']) or np.isnan(means['出弯']):
        return {'触发':False,'建议':'','证据':'重心/阶段数据不足',
                '依赖':['重心','阶段标签'],'规则置信度':'低'}

    shift = abs(means['入弯'] - means['出弯'])
    # 重心几乎不动(shift很小)→ 可能重心转移不足
    triggered = (shift < 0.02)   # 经验阈值(归一化坐标),待校准
    return {
        '触发': triggered,
        '建议': '入弯与出弯的重心位置变化很小,重心前后转移可能不足。健康的转弯中'
                '重心应随阶段流动(入弯压前、出弯移后)。可关注重心的主动前后移动。'
                if triggered else '',
        '证据': f'重心水平偏移 入弯{means["入弯"]:+.3f} 顶点{means["顶点"]:+.3f} '
                f'出弯{means["出弯"]:+.3f}(变化{shift:.3f})',
        '依赖': ['重心','阶段标签'],
        '规则置信度': '低',   # 前后重心矢状面量,2D+视角测不准
    }

res3 = rule_fore_aft(seq, iids, labels, trunk_s, quality)
print('===== 规则3:前后重心分阶段 =====')
print(f'触发: {res3["触发"]}')
print(f'证据: {res3["证据"]}')
print(f'规则置信度: {res3["规则置信度"]}   依赖: {res3["依赖"]}')
if res3['触发']:
    print(f'\n建议: {res3["建议"]}')
else:
    print('\n✅ 未触发(或重心转移正常)。')

In [ ]:
import numpy as np
from collections import defaultdict
from ski_physics import extract_all

# ========== 规则1:A-frame ==========
def rule_aframe(seq, iids, labels, trunk_s):
    KNEE_R, KNEE_L, ANK_R, ANK_L = 11, 14, 12, 15
    phase_af = defaultdict(list)
    for i, iid in enumerate(iids):
        ann = seq[iid]['annotation']
        kR,kL,aR,aL = [np.asarray(ann[x],float) for x in [KNEE_R,KNEE_L,ANK_R,ANK_L]]
        if any(p[2]<1 for p in [kR,kL,aR,aL]): continue
        kd = abs(kL[0]-kR[0]); fd = abs(aL[0]-aR[0])
        if kd < 1e-6: continue
        phase_af[labels[i]].append(fd/kd)
    init = np.mean(phase_af['入弯']) if phase_af['入弯'] else np.nan
    comp = np.mean(phase_af['出弯']) if phase_af['出弯'] else np.nan
    if np.isnan(init) or np.isnan(comp):
        return {'触发':False,'建议':'','证据':'数据不足','依赖':['膝踝距离'],'规则置信度':'中'}
    diff = init - comp
    triggered = (init > 1.2 and diff > 0.15)
    return {
        '触发': triggered,
        '建议': '入弯时脚比膝明显偏开,可能存在轻度A-frame倾向,提示内侧腿不够主动内倾。'
                '可尝试入弯时用内侧脚踝引导倾倒,让内侧膝更早跟上。' if triggered else '',
        '证据': f'入弯A-frame指标{init:.2f} vs 出弯{comp:.2f}(差{diff:+.2f})',
        '依赖': ['膝踝距离'],
        '规则置信度': '中',
    }

# ========== 规则2:内侧腿屈曲 ==========
def rule_inner_leg(seq, iids, labels, trunk_s, front_view=True):
    diffs = []
    for i in range(len(iids)):
        if labels[i] != '顶点': continue
        r = extract_all(seq[iids[i]]['annotation'])
        kr, ok_r = r['joints']['右膝']; kl, ok_l = r['joints']['左膝']
        if not (ok_r and ok_l): continue
        if not (90<=kr<=180 and 90<=kl<=180): continue
        tilt = trunk_s[i]
        if (tilt < 0) == front_view:   # 前拍:图像左倾=右弯=内侧右腿
            inner, outer = kr, kl
        else:
            inner, outer = kl, kr
        diffs.append(inner - outer)
    if not diffs:
        return {'触发':False,'建议':'','证据':'数据不足','依赖':['膝角','内倾方向'],'规则置信度':'低'}
    med = np.median(diffs)
    triggered = (med >= -3)
    return {
        '触发': triggered,
        '建议': '顶点附近内侧腿屈曲偏少,内侧腿应主动深屈收短,为身体内倾让空间、建立更大刃角。'
                if triggered else '',
        '证据': f'内侧-外侧膝角中位数{med:+.0f}°(共{len(diffs)}个顶点)',
        '依赖': ['膝角','内倾方向'],
        '规则置信度': '低',
    }

print('✅ 规则1、规则2 已重新定义')

### From metrics to advice with honest confidence

Each rule declares which metrics it depends on. The final confidence of any piece of advice is the minimum of the rule's own reliability and the quality of the data it depends on. This way, a viewpoint problem only weakens the rules that actually rely on the affected metric, while independent rules keep working normally.


In [ ]:
import numpy as np

def combine_confidence(rule_conf, dependencies, quality):
    """最终置信度 = min(规则本身, 数据对依赖量的置信度)"""
    level = {'高':3,'中':2,'低':1}
    final = level[rule_conf]; notes = []
    # 内倾方向不可信 → 拖累依赖它的规则
    if '内倾方向' in dependencies and not quality.get('viewpoint_ok', True):
        final = min(final,1); notes.append('视角问题使内倾方向不可信')
    # 阶段标签间接依赖内倾基线 → 若基线偏移大,降一档
    if '阶段标签' in dependencies and abs(quality['metrics'].get('内倾中位数',0))>8:
        final = min(final,2); notes.append('基线偏移使阶段标签欠可靠')
    # 遮挡率检查
    dep_metric = {'膝角':'膝角可信率','膝踝距离':'膝角可信率','脚髋距离':'膝角可信率',
                  '重心':'重心可信率','反弓':'反弓可信率'}
    for d in dependencies:
        if d in dep_metric and quality['metrics'].get(dep_metric[d],1.0)<0.5:
            final = min(final,1); notes.append(f'{d}遮挡严重')
    return {3:'高',2:'中',1:'低'}[final], notes


def run_expert_system(seq, iids, labels, trunk_s, quality):
    rules = [
        ('A-frame检测',    rule_aframe(seq, iids, labels, trunk_s)),
        ('内侧腿屈曲',     rule_inner_leg(seq, iids, labels, trunk_s)),
        ('前后重心分阶段', rule_fore_aft(seq, iids, labels, trunk_s, quality)),
        ('外侧腿反弓',     rule_angulation(seq, iids, labels, quality)),
        ('站姿宽度',       rule_stance_width(seq, iids, labels, quality)),
    ]

    print('='*58)
    print('           滑雪技术分析报告')
    print('='*58)
    print(f'\n【数据质量】总体置信度: {quality["overall"]}')
    for w in quality['warnings']:
        print(f'  {w}')
    if quality['overall']=='低':
        print('  → 以下建议请谨慎参考,建议用侧后方、相机水平的视频重拍。')

    print('\n【检测到的技术要点】')
    triggered_rules = []
    for name, res in rules:
        fc, notes = combine_confidence(res['规则置信度'], res['依赖'], quality)
        res['_final_conf'] = fc; res['_notes'] = notes
        if res['触发']:
            triggered_rules.append((name, res))

    if triggered_rules:
        for name, res in triggered_rules:
            print(f'\n● {name}  [置信度: {res["_final_conf"]}]')
            print(f'  {res["建议"]}')
            print(f'  依据: {res["证据"]}')
            if res['_notes']:
                print(f'  ⚠️ 置信度受限: {"; ".join(res["_notes"])}')
    else:
        print('\n  未检测到明显技术问题。')

    print('\n【未触发的检查(表现正常或数据不足)】')
    for name, res in rules:
        if not res['触发']:
            print(f'  ○ {name} [{res["_final_conf"]}]: {res["证据"]}')

    print('\n' + '='*58)
    print('注:本分析基于2D姿态估计,受拍摄视角影响。阈值为经验值,')
    print('   最终应结合专业教练意见。')
    print('='*58)


# ===== 运行完整引擎 =====
run_expert_system(seq, iids, labels, trunk_s, quality)

In [ ]:
import numpy as np

def rule_symmetry(seq, iids, labels, trunk_s, apexes, quality, front_view=True):
    """
    规则4:左右对称性。对比左弯组 vs 右弯组的顶点指标(反弓、内倾幅度)。
    依赖内倾方向分左右弯 → 当前数据受限。
    """
    from ski_physics import extract_all

    # 按方向给顶点分组
    left_turn = {'反弓':[], '内倾幅度':[]}   # 图像右倾(前拍=滑手左倾=左弯)
    right_turn = {'反弓':[], '内倾幅度':[]}  # 图像左倾(前拍=滑手右倾=右弯)

    for i in range(len(iids)):
        if labels[i] != '顶点': continue
        r = extract_all(seq[iids[i]]['annotation'])
        # 反弓(取较大者=外侧腿)
        vals=[]
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals=[v for v in vals if v<=60]
        angu = max(vals) if vals else None
        incl = abs(trunk_s[i])   # 内倾幅度

        tilt = trunk_s[i]
        # 前拍:图像左倾(tilt<0)=右弯;图像右倾(tilt>0)=左弯
        is_right = (tilt < 0) if front_view else (tilt > 0)
        grp = right_turn if is_right else left_turn
        if angu is not None: grp['反弓'].append(angu)
        grp['内倾幅度'].append(incl)

    n_left = len(left_turn['内倾幅度'])
    n_right = len(right_turn['内倾幅度'])

    # 需要两边都有足够的弯才能比
    if n_left < 2 or n_right < 2:
        return {'触发':False,'建议':'',
                '证据':f'左弯{n_left}个/右弯{n_right}个,一侧不足无法对比对称性',
                '依赖':['反弓','内倾方向'],'规则置信度':'低'}

    # 对比左右
    l_angu = np.median(left_turn['反弓']) if left_turn['反弓'] else np.nan
    r_angu = np.median(right_turn['反弓']) if right_turn['反弓'] else np.nan
    l_incl = np.median(left_turn['内倾幅度'])
    r_incl = np.median(right_turn['内倾幅度'])

    angu_diff = abs(l_angu - r_angu) if not (np.isnan(l_angu) or np.isnan(r_angu)) else np.nan
    incl_diff = abs(l_incl - r_incl)

    # 差异超过阈值 → 不对称
    triggered = (not np.isnan(angu_diff) and angu_diff > 10) or (incl_diff > 8)
    weaker = '左弯' if (l_incl < r_incl) else '右弯'
    return {
        '触发': triggered,
        '建议': f'左右转弯存在明显不对称({weaker}的倾斜/反弓偏弱)。'
                f'大多数人有强侧弱侧,建议针对性练习较弱的{weaker},'
                f'追求两侧动作一致。' if triggered else '',
        '证据': f'左弯(反弓{l_angu:.0f}°/内倾{l_incl:.0f}°) vs '
                f'右弯(反弓{r_angu:.0f}°/内倾{r_incl:.0f}°)',
        '依赖': ['反弓','内倾方向'],
        '规则置信度': '中',
    }

res4 = rule_symmetry(seq, iids, labels, trunk_s, apexes, quality)
print('===== 规则4:左右对称性 =====')
print(f'触发: {res4["触发"]}')
print(f'证据: {res4["证据"]}')
print(f'规则置信度: {res4["规则置信度"]}   依赖: {res4["依赖"]}')
if res4['触发']:
    print(f'\n建议: {res4["建议"]}')
else:
    print('\n○ 未触发或数据不足。')

In [ ]:
import numpy as np

def rule_rhythm(seq, iids, labels, trunk_s, apexes, quality):
    """
    规则7:动作流畅度/节奏。看转弯间隔是否均匀 + 内倾变化是否平滑。
    不依赖左右方向,主要看曲线的时间结构。
    """
    apex_frames = sorted([i for i, _ in apexes])

    result = {'依赖':['内倾幅度'], '规则置信度':'中'}

    # ---- 指标1:转弯节奏均匀度 ----
    rhythm_note = ''
    if len(apex_frames) >= 3:
        intervals = np.diff(apex_frames)   # 相邻顶点间隔
        cv = np.std(intervals) / np.mean(intervals)  # 变异系数(越小越均匀)
        rhythm_note = f'转弯间隔{intervals.tolist()}帧,变异系数{cv:.2f}'
        rhythm_bad = (cv > 0.6)   # 经验阈值:间隔波动大
    else:
        cv = np.nan; rhythm_bad = False
        rhythm_note = f'顶点数不足({len(apex_frames)}),无法评估节奏'

    # ---- 指标2:内倾变化平滑度 ----
    # 二阶差分的均方根,衡量曲线"加速度"大小
    d2 = np.diff(trunk_s, 2)
    roughness = np.sqrt(np.mean(d2**2))
    smooth_note = f'内倾曲线粗糙度{roughness:.2f}'
    rough_bad = (roughness > 2.0)   # 经验阈值

    triggered = rhythm_bad or rough_bad
    reasons = []
    if rhythm_bad: reasons.append('转弯节奏不均匀')
    if rough_bad: reasons.append('内倾变化不够平滑')

    result.update({
        '触发': triggered,
        '建议': f'动作流畅度可提升({"、".join(reasons)})。流畅的滑行应让身体连贯地'
                f'从一个弯过渡到下一个弯,节奏均匀、无明显卡顿。可尝试更主动、'
                f'连续地引导重心和倾斜的转换。' if triggered else '',
        '证据': f'{rhythm_note};{smooth_note}',
    })
    return result

res7 = rule_rhythm(seq, iids, labels, trunk_s, apexes, quality)
print('===== 规则7:动作流畅度/节奏 =====')
print(f'触发: {res7["触发"]}')
print(f'证据: {res7["证据"]}')
print(f'规则置信度: {res7["规则置信度"]}   依赖: {res7["依赖"]}')
if res7['触发']:
    print(f'\n建议: {res7["建议"]}')
else:
    print('\n✅ 动作流畅,未触发。')

In [ ]:
import numpy as np

def rule_vertical_separation(seq, iids, labels, quality):
    """
    规则8:垂直分离(两脚高低差),顶点时应明显。
    用两踝y坐标差/髋宽归一化。不依赖左右方向。
    """
    from ski_physics import extract_all
    ANK_R, ANK_L, HIP_R, HIP_L = 12, 15, 10, 13
    VSEP_MIN = 0.3   # 经验阈值:高低差/髋宽 <0.3 算垂直分离不足(待校准)

    ratios = []
    for i in range(len(iids)):
        if labels[i] != '顶点': continue
        ann = seq[iids[i]]['annotation']
        aR,aL,hR,hL = [np.asarray(ann[x],float) for x in [ANK_R,ANK_L,HIP_R,HIP_L]]
        if any(p[2]<1 for p in [aR,aL,hR,hL]): continue
        vsep = abs(aL[1]-aR[1])       # 两踝高低差(y方向)
        hip_w = abs(hL[0]-hR[0])       # 髋宽做归一化尺度
        if hip_w < 1e-6: continue
        ratio = vsep / hip_w
        if ratio > 5: continue         # 异常过滤
        ratios.append(ratio)

    if not ratios:
        return {'触发':False,'建议':'','证据':'顶点无可信垂直分离数据',
                '依赖':['脚髋距离'],'规则置信度':'中'}

    med = np.median(ratios)
    triggered = (med < VSEP_MIN)
    return {
        '触发': triggered,
        '建议': f'顶点附近垂直分离偏小(两脚高低差约为髋宽的{med:.1f}倍)。'
                f'刻滑时内外侧腿应一屈一伸,产生明显高低差以建立立刃。'
                f'可加强内侧腿屈曲、外侧腿伸展,增大垂直分离。' if triggered else '',
        '证据': f'顶点 两脚高低差/髋宽 中位数{med:.2f}(共{len(ratios)}帧,阈值{VSEP_MIN})',
        '依赖': ['脚髋距离'],
        '规则置信度': '中',
    }

res8 = rule_vertical_separation(seq, iids, labels, quality)
print('===== 规则8:垂直分离 =====')
print(f'触发: {res8["触发"]}')
print(f'证据: {res8["证据"]}')
print(f'规则置信度: {res8["规则置信度"]}   依赖: {res8["依赖"]}')
if res8['触发']:
    print(f'\n建议: {res8["建议"]}')
else:
    print('\n✅ 垂直分离充分,未触发。')

In [ ]:
import numpy as np

def rule_flexion(seq, iids, labels, quality):
    """
    规则9:身体屈伸循环。重心高度应随阶段起伏(顶点低/过渡高)。
    y轴朝下:y大=重心低,y小=重心高。不依赖左右方向。
    """
    from ski_physics import extract_all

    phase_y = {'顶点':[], '过渡':[]}
    all_y = []
    for i in range(len(iids)):
        r = extract_all(seq[iids[i]]['annotation'])
        cx, cy, mcov, ok = r['com']
        if not ok: continue
        all_y.append(cy)
        if labels[i] in phase_y:
            phase_y[labels[i]].append(cy)

    if len(all_y) < 5:
        return {'触发':False,'建议':'','证据':'重心数据不足',
                '依赖':['重心','阶段标签'],'规则置信度':'低'}

    # 指标1:整体起伏幅度(重心高度的波动)
    y_range = np.max(all_y) - np.min(all_y)

    # 指标2:顶点 vs 过渡的高度差(顶点应更低=y更大)
    apex_y = np.median(phase_y['顶点']) if phase_y['顶点'] else np.nan
    trans_y = np.median(phase_y['过渡']) if phase_y['过渡'] else np.nan

    # 判断:起伏太小(僵直)→ 触发
    RANGE_MIN = 0.05   # 经验阈值:重心高度波动<0.05(归一化)算起伏不足
    triggered = (y_range < RANGE_MIN)

    # 附带说明屈伸方向对不对
    dir_note = ''
    if not (np.isnan(apex_y) or np.isnan(trans_y)):
        if apex_y > trans_y:   # 顶点y更大=顶点更低,符合规律
            dir_note = '顶点低/过渡高,屈伸方向正确'
        else:
            dir_note = '顶点未明显低于过渡,屈伸循环可能不清晰'

    return {
        '触发': triggered,
        '建议': f'身体屈伸幅度偏小(重心高度起伏{y_range:.2f}),可能腿部主动屈伸不足。'
                f'刻滑应像减震器:顶点屈膝吸收压力、过渡伸展换刃,让身体高度规律起伏。'
                if triggered else '',
        '证据': f'重心高度起伏{y_range:.2f}(阈值{RANGE_MIN});{dir_note}',
        '依赖': ['重心','阶段标签'],
        '规则置信度': '中',
    }

res9 = rule_flexion(seq, iids, labels, quality)
print('===== 规则9:身体屈伸循环 =====')
print(f'触发: {res9["触发"]}')
print(f'证据: {res9["证据"]}')
print(f'规则置信度: {res9["规则置信度"]}   依赖: {res9["依赖"]}')
if res9['触发']:
    print(f'\n建议: {res9["建议"]}')
else:
    print('\n✅ 屈伸循环正常,未触发。')

In [ ]:
import numpy as np
from ski_physics import extract_all

# ========== 规则10:内倾充分性(顶点内倾够不够大)==========
def rule_inclination(seq, iids, labels, trunk_s, quality):
    """顶点时内倾幅度是否充分。内倾不足=没真正立刃过弯。不依赖方向(只看幅度)。"""
    INCL_MIN = 12   # 经验阈值:顶点内倾<12°算偏小
    apex_incl = [abs(trunk_s[i]) for i in range(len(iids)) if labels[i]=='顶点']
    if not apex_incl:
        return {'触发':False,'建议':'','证据':'无顶点数据','依赖':['内倾幅度'],'规则置信度':'中'}
    med = np.median(apex_incl)
    triggered = (med < INCL_MIN)
    return {
        '触发': triggered,
        '建议': f'顶点内倾偏小(中位数{med:.0f}°),身体倾斜不足可能导致立刃不够、抓地弱。'
                f'可尝试过弯时让身体更多倒向弯心,增大刃角。' if triggered else '',
        '证据': f'顶点内倾幅度中位数{med:.0f}°(阈值{INCL_MIN}°,共{len(apex_incl)}顶点)',
        '依赖': ['内倾幅度'], '规则置信度': '中',
    }

# ========== 规则11:换刃过渡速度(过渡阶段是否拖沓)==========
def rule_transition(seq, iids, labels, quality):
    """过渡阶段占比。过渡太长=换刃拖沓、两弯间'发呆'。不依赖方向。"""
    n_trans = sum(1 for l in labels if l=='过渡')
    n_total = len(labels)
    ratio = n_trans / n_total
    TRANS_MAX = 0.35   # 经验:过渡占比>35%算换刃偏慢
    triggered = (ratio > TRANS_MAX)
    return {
        '触发': triggered,
        '建议': f'过渡阶段占比偏高({ratio:.0%}),换刃可能偏慢或两弯衔接拖沓。'
                f'流畅滑行应快速完成换刃,减少"直立发呆"的时间。' if triggered else '',
        '证据': f'过渡帧占比{ratio:.0%}({n_trans}/{n_total},阈值{TRANS_MAX:.0%})',
        '依赖': ['阶段标签'], '规则置信度': '中',
    }

# ========== 规则12:反弓分阶段(顶点反弓 vs 入弯反弓)==========
def rule_angulation_phased(seq, iids, labels, quality):
    """反弓应在顶点最大。若顶点反弓不比入弯明显大,说明反弓建立时机/幅度有问题。"""
    phase_angu = {'入弯':[], '顶点':[]}
    for i in range(len(iids)):
        if labels[i] not in phase_angu: continue
        r = extract_all(seq[iids[i]]['annotation'])
        vals=[]
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals=[v for v in vals if v<=60]
        if vals: phase_angu[labels[i]].append(max(vals))
    a_init = np.median(phase_angu['入弯']) if phase_angu['入弯'] else np.nan
    a_apex = np.median(phase_angu['顶点']) if phase_angu['顶点'] else np.nan
    if np.isnan(a_init) or np.isnan(a_apex):
        return {'触发':False,'建议':'','证据':'入弯或顶点反弓数据不足','依赖':['反弓','阶段标签'],'规则置信度':'中'}
    triggered = (a_apex <= a_init)   # 顶点反弓没比入弯大
    return {
        '触发': triggered,
        '建议': f'反弓在顶点未明显增大(入弯{a_init:.0f}°→顶点{a_apex:.0f}°)。反弓应随'
                f'过弯逐渐建立、在顶点最大。可关注在弯的后半段主动加强上下身分离。'
                if triggered else '',
        '证据': f'反弓 入弯{a_init:.0f}°→顶点{a_apex:.0f}°',
        '依赖': ['反弓','阶段标签'], '规则置信度': '中',
    }

# 跑这三条
for name, fn in [('规则10 内倾充分性', rule_inclination(seq,iids,labels,trunk_s,quality)),
                 ('规则11 换刃过渡速度', rule_transition(seq,iids,labels,quality)),
                 ('规则12 反弓分阶段', rule_angulation_phased(seq,iids,labels,quality))]:
    print(f'===== {name} =====')
    print(f'  触发:{fn["触发"]}  置信度:{fn["规则置信度"]}')
    print(f'  证据:{fn["证据"]}')
    if fn['触发']: print(f'  建议:{fn["建议"]}')
    print()

In [ ]:
import numpy as np
from ski_physics import extract_all

# ========== 规则13:身体稳定性(顶点内倾一致性)==========
def rule_consistency(seq, iids, labels, trunk_s, apexes, quality):
    """各次转弯的顶点内倾是否一致。忽大忽小=转弯质量不稳定。"""
    apex_incl = [abs(trunk_s[i]) for i,_ in apexes]
    if len(apex_incl) < 3:
        return {'触发':False,'建议':'','证据':'顶点数不足','依赖':['内倾幅度'],'规则置信度':'中'}
    cv = np.std(apex_incl)/np.mean(apex_incl)   # 变异系数
    triggered = (cv > 0.4)
    return {
        '触发': triggered,
        '建议': f'各次转弯的倾斜程度不一致(变异{cv:.2f}),转弯质量不稳定。'
                f'可追求每个弯的一致性,建立稳定的节奏和刃角。' if triggered else '',
        '证据': f'顶点内倾变异系数{cv:.2f}(共{len(apex_incl)}弯,阈值0.4)',
        '依赖': ['内倾幅度'], '规则置信度': '中',
    }

# ========== 规则14:膝关节屈曲整体水平(是否太直/太蹲)==========
def rule_knee_flexion(seq, iids, labels, quality):
    """顶点整体膝角。太直(接近180)=没吃压力;太蹲(<90过滤后仍很小)=过度。"""
    knees = []
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        r = extract_all(seq[iids[i]]['annotation'])
        for k in ['右膝','左膝']:
            ang, ok = r['joints'][k]
            if ok and 90<=ang<=180: knees.append(ang)
    if not knees:
        return {'触发':False,'建议':'','证据':'膝角数据不足','依赖':['膝角'],'规则置信度':'中'}
    med = np.median(knees)
    triggered = (med > 165)   # 顶点膝角太直
    return {
        '触发': triggered,
        '建议': f'顶点膝关节偏直(中位数{med:.0f}°),腿部屈曲不足,难以吸收压力、控制刃角。'
                f'可加强屈膝,用腿部主动承压。' if triggered else '',
        '证据': f'顶点膝角中位数{med:.0f}°(阈值165°)',
        '依赖': ['膝角'], '规则置信度': '中',
    }

# ========== 规则15:重心横向移动幅度(是否有效跨越)==========
def rule_com_lateral(seq, iids, labels, quality):
    """重心横向(x)移动范围。换弯时重心要跨过雪板到新弯内侧,移动太小=换弯不彻底。"""
    xs = []
    for i in range(len(iids)):
        r = extract_all(seq[iids[i]]['annotation'])
        cx,cy,mcov,ok = r['com']
        if ok: xs.append(cx)
    if len(xs) < 5:
        return {'触发':False,'建议':'','证据':'重心数据不足','依赖':['重心'],'规则置信度':'中'}
    x_range = np.max(xs)-np.min(xs)
    triggered = (x_range < 0.1)   # 横向移动太小
    return {
        '触发': triggered,
        '建议': f'重心横向移动幅度偏小({x_range:.2f}),换弯时重心可能没有充分跨越到'
                f'新弯内侧。可加强换弯时重心的主动横移(cross-over)。' if triggered else '',
        '证据': f'重心横向移动范围{x_range:.2f}(阈值0.1)',
        '依赖': ['重心'], '规则置信度': '中',
    }

# ========== 规则16:内倾-反弓配比(倾斜靠倒身体还是靠反弓)==========
def rule_incl_angu_ratio(seq, iids, labels, trunk_s, quality):
    """顶点时,身体倾斜主要靠'整体倒'(内倾大反弓小)还是'上下分离'(反弓大)。
       理想是有足够反弓,而非纯靠倒身体。"""
    ratios = []
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        r = extract_all(seq[iids[i]]['annotation'])
        vals=[]
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals=[v for v in vals if v<=60]
        if not vals: continue
        angu = max(vals); incl = abs(trunk_s[i])
        if incl < 1: continue
        ratios.append(angu/incl)   # 反弓/内倾,越大越靠反弓
    if not ratios:
        return {'触发':False,'建议':'','证据':'数据不足','依赖':['反弓','内倾幅度'],'规则置信度':'中'}
    med = np.median(ratios)
    triggered = (med < 0.5)   # 反弓远小于内倾=纯靠倒身体
    return {
        '触发': triggered,
        '建议': f'过弯主要靠整体倒向弯内、反弓占比偏低(反弓/内倾{med:.1f})。'
                f'可加强上下身分离(反弓),用下肢内倾而非整个身体倒来立刃。' if triggered else '',
        '证据': f'反弓/内倾比中位数{med:.1f}(阈值0.5)',
        '依赖': ['反弓','内倾幅度'], '规则置信度': '中',
    }

# 跑这批
for name, fn in [('规则13 转弯一致性', rule_consistency(seq,iids,labels,trunk_s,apexes,quality)),
                 ('规则14 膝屈曲水平', rule_knee_flexion(seq,iids,labels,quality)),
                 ('规则15 重心横移', rule_com_lateral(seq,iids,labels,quality)),
                 ('规则16 内倾反弓配比', rule_incl_angu_ratio(seq,iids,labels,trunk_s,quality))]:
    print(f'===== {name} =====  触发:{fn["触发"]} 置信度:{fn["规则置信度"]}')
    print(f'  证据:{fn["证据"]}')
    if fn['触发']: print(f'  建议:{fn["建议"]}')
    print()

In [ ]:
%%writefile /content/drive/MyDrive/ski_app/ski_expert.py
"""
ski_expert.py — 双板滑雪专家系统(第五关)
输入:每帧物理量(经ski_physics) + 阶段标签 + 内倾曲线
输出:带置信度的技术分析报告
依赖:ski_physics.py (同目录)
"""
import numpy as np
from collections import defaultdict
import ski_physics
from ski_physics import extract_all

# ========================================================
# 数据质量门控
# ========================================================
def assess_data_quality(seq, iids, trunk_s, apexes):
    T = len(iids); report = {'warnings': [], 'metrics': {}}
    tc = {'内倾':0, '重心':0, '膝角':0, '反弓':0}
    for iid in iids:
        r = extract_all(seq[iid]['annotation'])
        if r['trunk_tilt'][1]: tc['内倾'] += 1
        if r['com'][3]: tc['重心'] += 1
        if r['joints']['右膝'][1] and r['joints']['左膝'][1]: tc['膝角'] += 1
        if r['angulation_R'][1]: tc['反弓'] += 1
    for k, v in tc.items():
        rate = v / T; report['metrics'][f'{k}可信率'] = rate
        if rate < 0.5: report['warnings'].append(f'⚠️ {k}可信率仅{rate:.0%},遮挡严重')
    signs = [np.sign(trunk_s[i]) for i, _ in apexes]
    if signs:
        pos = sum(1 for s in signs if s > 0); neg = sum(1 for s in signs if s < 0)
        tot = len(signs); mr = min(pos, neg)/tot if tot else 0
        report['metrics']['左右弯平衡度'] = mr
        if mr < 0.2:
            report['warnings'].append(
                f'⚠️ 转弯几乎全同方向({max(pos,neg)}/{tot})。可能:(a)视角问题使内倾方向失真;'
                f'或(b)真实同向弯为主。建议确认拍摄角度或换侧后方、相机水平的视频。'
                f'内倾【方向】置信度低,内倾【幅度】仍可参考。')
            report['viewpoint_ok'] = False
        else:
            report['viewpoint_ok'] = True
    mt = np.median(trunk_s); report['metrics']['内倾中位数'] = mt
    if abs(mt) > 8:
        report['warnings'].append(f'⚠️ 内倾中位数{mt:+.1f}°偏离0,存在基线偏移,内倾方向需谨慎。')
    report['overall'] = '低' if not report.get('viewpoint_ok', True) else ('中' if len(report['warnings'])>=2 else '高')
    return report

# ========================================================
# 16条规则(每条返回统一格式dict)
# ========================================================
def rule_aframe(seq, iids, labels, trunk_s):
    KR,KL,AR,AL = 11,14,12,15
    pa = defaultdict(list)
    for i, iid in enumerate(iids):
        a = seq[iid]['annotation']
        kR,kL,aR,aL = [np.asarray(a[x],float) for x in [KR,KL,AR,AL]]
        if any(p[2]<1 for p in [kR,kL,aR,aL]): continue
        kd=abs(kL[0]-kR[0]); fd=abs(aL[0]-aR[0])
        if kd<1e-6: continue
        pa[labels[i]].append(fd/kd)
    ini=np.mean(pa['入弯']) if pa['入弯'] else np.nan
    com=np.mean(pa['出弯']) if pa['出弯'] else np.nan
    if np.isnan(ini) or np.isnan(com):
        return {'触发':False,'建议':'','证据':'数据不足','依赖':['膝踝距离'],'规则置信度':'中'}
    d=ini-com; t=(ini>1.2 and d>0.15)
    return {'触发':t,'建议':'入弯时脚比膝明显偏开,可能存在轻度A-frame,提示内侧腿不够主动内倾。可用内侧脚踝引导倾倒,让内侧膝更早跟上。' if t else '',
            '证据':f'入弯A-frame指标{ini:.2f} vs 出弯{com:.2f}(差{d:+.2f})','依赖':['膝踝距离'],'规则置信度':'中'}

def rule_inner_leg(seq, iids, labels, trunk_s, front_view=True):
    diffs=[]
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        r=extract_all(seq[iids[i]]['annotation'])
        kr,okr=r['joints']['右膝']; kl,okl=r['joints']['左膝']
        if not(okr and okl): continue
        if not(90<=kr<=180 and 90<=kl<=180): continue
        inner,outer=(kr,kl) if (trunk_s[i]<0)==front_view else (kl,kr)
        diffs.append(inner-outer)
    if not diffs:
        return {'触发':False,'建议':'','证据':'数据不足','依赖':['膝角','内倾方向'],'规则置信度':'低'}
    m=np.median(diffs); t=(m>=-3)
    return {'触发':t,'建议':'顶点内侧腿屈曲偏少,应主动深屈收短,为身体内倾让空间、建立更大刃角。' if t else '',
            '证据':f'内侧-外侧膝角中位数{m:+.0f}°(共{len(diffs)}顶点)','依赖':['膝角','内倾方向'],'规则置信度':'低'}

def rule_fore_aft(seq, iids, labels, trunk_s):
    AR,AL=12,15; po={'入弯':[],'顶点':[],'出弯':[]}
    for i in range(len(iids)):
        if labels[i] not in po: continue
        r=extract_all(seq[iids[i]]['annotation'])
        cx,cy,mc,ok=r['com']
        if not ok: continue
        a=seq[iids[i]]['annotation']; aR,aL=np.asarray(a[AR],float),np.asarray(a[AL],float)
        if aR[2]<1 or aL[2]<1: continue
        po[labels[i]].append(cx-(aR[0]+aL[0])/2)
    mn={k:(np.median(v) if v else np.nan) for k,v in po.items()}
    if np.isnan(mn['入弯']) or np.isnan(mn['出弯']):
        return {'触发':False,'建议':'','证据':'重心/阶段数据不足','依赖':['重心','阶段标签'],'规则置信度':'低'}
    sh=abs(mn['入弯']-mn['出弯']); t=(sh<0.02)
    return {'触发':t,'建议':'入弯与出弯重心位置变化很小,前后转移可能不足。健康转弯中重心应随阶段流动。' if t else '',
            '证据':f'重心偏移 入弯{mn["入弯"]:+.3f} 顶点{mn["顶点"]:+.3f} 出弯{mn["出弯"]:+.3f}(变化{sh:.3f})',
            '依赖':['重心','阶段标签'],'规则置信度':'低'}

def rule_symmetry(seq, iids, labels, trunk_s, front_view=True):
    lt={'反弓':[],'内倾':[]}; rt={'反弓':[],'内倾':[]}
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        r=extract_all(seq[iids[i]]['annotation']); vals=[]
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals=[v for v in vals if v<=60]; angu=max(vals) if vals else None
        g=rt if ((trunk_s[i]<0)==front_view) else lt
        if angu is not None: g['反弓'].append(angu)
        g['内倾'].append(abs(trunk_s[i]))
    nl,nr=len(lt['内倾']),len(rt['内倾'])
    if nl<2 or nr<2:
        return {'触发':False,'建议':'','证据':f'左弯{nl}/右弯{nr},一侧不足','依赖':['反弓','内倾方向'],'规则置信度':'低'}
    la,ra=np.median(lt['反弓']) if lt['反弓'] else np.nan, np.median(rt['反弓']) if rt['反弓'] else np.nan
    li,ri=np.median(lt['内倾']),np.median(rt['内倾'])
    ad=abs(la-ra) if not(np.isnan(la)or np.isnan(ra)) else np.nan; idf=abs(li-ri)
    t=(not np.isnan(ad) and ad>10) or (idf>8); w='左弯' if li<ri else '右弯'
    return {'触发':t,'建议':f'左右转弯不对称({w}偏弱),建议针对性练习较弱侧。' if t else '',
            '证据':f'左弯(反弓{la:.0f}/内倾{li:.0f}) vs 右弯(反弓{ra:.0f}/内倾{ri:.0f})','依赖':['反弓','内倾方向'],'规则置信度':'中'}

def rule_angulation(seq, iids, labels):
    aa=[]
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        r=extract_all(seq[iids[i]]['annotation']); vals=[]
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals=[v for v in vals if v<=60]
        if vals: aa.append(max(vals))
    if not aa:
        return {'触发':False,'建议':'','证据':'顶点无可信反弓','依赖':['反弓'],'规则置信度':'中'}
    m=np.median(aa); t=(m<10)
    return {'触发':t,'建议':'顶点外侧腿反弓偏小,可能靠整体倒向弯内而非上下身分离。可上身更直立、下肢独立内倾。' if t else '',
            '证据':f'顶点外侧腿反弓中位数{m:.0f}°(共{len(aa)}顶点,阈值10°)','依赖':['反弓'],'规则置信度':'中'}

def rule_stance_width(seq, iids, labels):
    HR,HL,AR,AL=10,13,12,15; rs=[]
    for i in range(len(iids)):
        if labels[i]!='过渡': continue
        a=seq[iids[i]]['annotation']; hR,hL,aR,aL=[np.asarray(a[x],float) for x in [HR,HL,AR,AL]]
        if any(p[2]<1 for p in [hR,hL,aR,aL]): continue
        hw=abs(hL[0]-hR[0]); fw=abs(aL[0]-aR[0])
        if hw<1e-6: continue
        rt=fw/hw
        if rt<=5: rs.append(rt)
    if not rs:
        return {'触发':False,'建议':'','证据':'过渡阶段无数据','依赖':['脚髋距离'],'规则置信度':'中'}
    m=np.median(rs); t=(m>1.4)
    return {'触发':t,'建议':f'过渡站姿偏宽(脚约髋宽{m:.1f}倍),会限制换刃灵活性。可收窄至与髋同宽。' if t else '',
            '证据':f'过渡 脚间距/髋宽 中位数{m:.2f}(共{len(rs)}帧,阈值1.4)','依赖':['脚髋距离'],'规则置信度':'中'}

def rule_rhythm(seq, iids, labels, trunk_s, apexes):
    af=sorted([i for i,_ in apexes]); note=''
    if len(af)>=3:
        iv=np.diff(af); cv=np.std(iv)/np.mean(iv); note=f'间隔{iv.tolist()},变异{cv:.2f}'; rb=(cv>0.6)
    else: cv=np.nan; rb=False; note=f'顶点不足({len(af)})'
    d2=np.diff(trunk_s,2); rough=np.sqrt(np.mean(d2**2)); rgb=(rough>2.0)
    t=rb or rgb; rs=[]
    if rb: rs.append('节奏不均匀')
    if rgb: rs.append('内倾变化不平滑')
    return {'触发':t,'建议':f'流畅度可提升({"、".join(rs)})。应连贯过渡、节奏均匀。' if t else '',
            '证据':f'{note};粗糙度{rough:.2f}','依赖':['内倾幅度'],'规则置信度':'中'}

def rule_vertical_separation(seq, iids, labels):
    AR,AL,HR,HL=12,15,10,13; rs=[]
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        a=seq[iids[i]]['annotation']; aR,aL,hR,hL=[np.asarray(a[x],float) for x in [AR,AL,HR,HL]]
        if any(p[2]<1 for p in [aR,aL,hR,hL]): continue
        vs=abs(aL[1]-aR[1]); hw=abs(hL[0]-hR[0])
        if hw<1e-6: continue
        rt=vs/hw
        if rt<=5: rs.append(rt)
    if not rs:
        return {'触发':False,'建议':'','证据':'顶点无数据','依赖':['脚髋距离'],'规则置信度':'中'}
    m=np.median(rs); t=(m<0.3)
    return {'触发':t,'建议':f'顶点垂直分离偏小(高低差约髋宽{m:.1f}倍),内外腿屈伸不足。可加强一屈一伸。' if t else '',
            '证据':f'顶点 高低差/髋宽 中位数{m:.2f}(共{len(rs)}帧,阈值0.3)','依赖':['脚髋距离'],'规则置信度':'中'}

def rule_flexion(seq, iids, labels):
    py={'顶点':[],'过渡':[]}; ay=[]
    for i in range(len(iids)):
        r=extract_all(seq[iids[i]]['annotation']); cx,cy,mc,ok=r['com']
        if not ok: continue
        ay.append(cy)
        if labels[i] in py: py[labels[i]].append(cy)
    if len(ay)<5:
        return {'触发':False,'建议':'','证据':'重心数据不足','依赖':['重心','阶段标签'],'规则置信度':'低'}
    yr=np.max(ay)-np.min(ay); apy=np.median(py['顶点']) if py['顶点'] else np.nan; try=np.median(py['过渡']) if py['过渡'] else np.nan
    t=(yr<0.05); dn=''
    if not(np.isnan(apy)or np.isnan(try)): dn='顶点低/过渡高,方向正确' if apy>try else '顶点未明显低于过渡,屈伸不清晰'
    return {'触发':t,'建议':f'屈伸幅度偏小(起伏{yr:.2f}),腿部主动屈伸不足。应顶点屈、过渡伸,规律起伏。' if t else '',
            '证据':f'重心高度起伏{yr:.2f}(阈值0.05);{dn}','依赖':['重心','阶段标签'],'规则置信度':'中'}

def rule_inclination(seq, iids, labels, trunk_s):
    ai=[abs(trunk_s[i]) for i in range(len(iids)) if labels[i]=='顶点']
    if not ai:
        return {'触发':False,'建议':'','证据':'无顶点数据','依赖':['内倾幅度'],'规则置信度':'中'}
    m=np.median(ai); t=(m<12)
    return {'触发':t,'建议':f'顶点内倾偏小({m:.0f}°),立刃可能不足。可让身体更多倒向弯心。' if t else '',
            '证据':f'顶点内倾中位数{m:.0f}°(阈值12°,共{len(ai)}顶点)','依赖':['内倾幅度'],'规则置信度':'中'}

def rule_transition(seq, iids, labels):
    nt=sum(1 for l in labels if l=='过渡'); n=len(labels); r=nt/n; t=(r>0.35)
    return {'触发':t,'建议':f'过渡占比偏高({r:.0%}),换刃可能拖沓。应快速换刃。' if t else '',
            '证据':f'过渡帧占比{r:.0%}({nt}/{n},阈值35%)','依赖':['阶段标签'],'规则置信度':'中'}

def rule_angulation_phased(seq, iids, labels):
    pa={'入弯':[],'顶点':[]}
    for i in range(len(iids)):
        if labels[i] not in pa: continue
        r=extract_all(seq[iids[i]]['annotation']); vals=[]
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals=[v for v in vals if v<=60]
        if vals: pa[labels[i]].append(max(vals))
    ai=np.median(pa['入弯']) if pa['入弯'] else np.nan; ap=np.median(pa['顶点']) if pa['顶点'] else np.nan
    if np.isnan(ai) or np.isnan(ap):
        return {'触发':False,'建议':'','证据':'入弯/顶点反弓不足','依赖':['反弓','阶段标签'],'规则置信度':'中'}
    t=(ap<=ai)
    return {'触发':t,'建议':f'反弓在顶点未明显增大(入弯{ai:.0f}→顶点{ap:.0f}),应随过弯建立、顶点最大。' if t else '',
            '证据':f'反弓 入弯{ai:.0f}°→顶点{ap:.0f}°','依赖':['反弓','阶段标签'],'规则置信度':'中'}

def rule_consistency(seq, iids, labels, trunk_s, apexes):
    ai=[abs(trunk_s[i]) for i,_ in apexes]
    if len(ai)<3:
        return {'触发':False,'建议':'','证据':'顶点不足','依赖':['内倾幅度'],'规则置信度':'中'}
    cv=np.std(ai)/np.mean(ai); t=(cv>0.4)
    return {'触发':t,'建议':f'各弯倾斜不一致(变异{cv:.2f}),质量不稳定。追求每弯一致。' if t else '',
            '证据':f'顶点内倾变异{cv:.2f}(共{len(ai)}弯,阈值0.4)','依赖':['内倾幅度'],'规则置信度':'中'}

def rule_knee_flexion(seq, iids, labels):
    ks=[]
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        r=extract_all(seq[iids[i]]['annotation'])
        for k in ['右膝','左膝']:
            a,ok=r['joints'][k]
            if ok and 90<=a<=180: ks.append(a)
    if not ks:
        return {'触发':False,'建议':'','证据':'膝角不足','依赖':['膝角'],'规则置信度':'中'}
    m=np.median(ks); t=(m>165)
    return {'触发':t,'建议':f'顶点膝偏直({m:.0f}°),屈曲不足难承压。加强屈膝。' if t else '',
            '证据':f'顶点膝角中位数{m:.0f}°(阈值165°)','依赖':['膝角'],'规则置信度':'中'}

def rule_com_lateral(seq, iids, labels):
    xs=[]
    for i in range(len(iids)):
        r=extract_all(seq[iids[i]]['annotation']); cx,cy,mc,ok=r['com']
        if ok: xs.append(cx)
    if len(xs)<5:
        return {'触发':False,'建议':'','证据':'重心不足','依赖':['重心'],'规则置信度':'中'}
    xr=np.max(xs)-np.min(xs); t=(xr<0.1)
    return {'触发':t,'建议':f'重心横移偏小({xr:.2f}),换弯cross-over可能不足。加强重心横移。' if t else '',
            '证据':f'重心横移范围{xr:.2f}(阈值0.1)','依赖':['重心'],'规则置信度':'中'}

def rule_incl_angu_ratio(seq, iids, labels, trunk_s):
    rs=[]
    for i in range(len(iids)):
        if labels[i]!='顶点': continue
        r=extract_all(seq[iids[i]]['annotation']); vals=[]
        if r['angulation_R'][1]: vals.append(abs(r['angulation_R'][0]))
        if r['angulation_L'][1]: vals.append(abs(r['angulation_L'][0]))
        vals=[v for v in vals if v<=60]
        if not vals: continue
        angu=max(vals); incl=abs(trunk_s[i])
        if incl<1: continue
        rs.append(angu/incl)
    if not rs:
        return {'触发':False,'建议':'','证据':'数据不足','依赖':['反弓','内倾幅度'],'规则置信度':'中'}
    m=np.median(rs); t=(m<0.5)
    return {'触发':t,'建议':f'过弯主要靠整体倒(反弓/内倾{m:.1f}),反弓占比低。加强上下身分离。' if t else '',
            '证据':f'反弓/内倾比中位数{m:.1f}(阈值0.5)','依赖':['反弓','内倾幅度'],'规则置信度':'中'}

# ========================================================
# 置信度传导 + 主引擎
# ========================================================
def combine_confidence(rule_conf, deps, quality):
    lv={'高':3,'中':2,'低':1}; f=lv[rule_conf]; notes=[]
    if '内倾方向' in deps and not quality.get('viewpoint_ok',True):
        f=min(f,1); notes.append('视角问题使内倾方向不可信')
    if '阶段标签' in deps and abs(quality['metrics'].get('内倾中位数',0))>8:
        f=min(f,2); notes.append('基线偏移使阶段标签欠可靠')
    dm={'膝角':'膝角可信率','膝踝距离':'膝角可信率','脚髋距离':'膝角可信率','重心':'重心可信率','反弓':'反弓可信率'}
    for d in deps:
        if d in dm and quality['metrics'].get(dm[d],1.0)<0.5:
            f=min(f,1); notes.append(f'{d}遮挡严重')
    return {3:'高',2:'中',1:'低'}[f], notes

def run_expert_system(seq, iids, labels, trunk_s, apexes, quality, verbose=True):
    rules=[
        ('A-frame检测', rule_aframe(seq,iids,labels,trunk_s)),
        ('内侧腿屈曲', rule_inner_leg(seq,iids,labels,trunk_s)),
        ('前后重心分阶段', rule_fore_aft(seq,iids,labels,trunk_s)),
        ('左右对称性', rule_symmetry(seq,iids,labels,trunk_s)),
        ('外侧腿反弓', rule_angulation(seq,iids,labels)),
        ('站姿宽度', rule_stance_width(seq,iids,labels)),
        ('动作流畅度', rule_rhythm(seq,iids,labels,trunk_s,apexes)),
        ('垂直分离', rule_vertical_separation(seq,iids,labels)),
        ('屈伸循环', rule_flexion(seq,iids,labels)),
        ('内倾充分性', rule_inclination(seq,iids,labels,trunk_s)),
        ('换刃过渡速度', rule_transition(seq,iids,labels)),
        ('反弓分阶段', rule_angulation_phased(seq,iids,labels)),
        ('转弯一致性', rule_consistency(seq,iids,labels,trunk_s,apexes)),
        ('膝屈曲水平', rule_knee_flexion(seq,iids,labels)),
        ('重心横移', rule_com_lateral(seq,iids,labels)),
        ('内倾反弓配比', rule_incl_angu_ratio(seq,iids,labels,trunk_s)),
    ]
    for name,res in rules:
        res['_conf'],res['_notes']=combine_confidence(res['规则置信度'],res['依赖'],quality)
    if verbose:
        print('='*60); print('              滑雪技术分析报告'); print('='*60)
        print(f'\n【数据质量】总体置信度: {quality["overall"]}')
        for w in quality['warnings']: print(f'  {w}')
        print('\n【检测到的技术要点】')
        trig=[(n,r) for n,r in rules if r['触发']]
        if trig:
            for n,r in trig:
                print(f'\n● {n}  [置信度: {r["_conf"]}]')
                print(f'  {r["建议"]}')
                print(f'  依据: {r["证据"]}')
                if r['_notes']: print(f'  ⚠️ {"; ".join(r["_notes"])}')
        else: print('  未检测到明显技术问题。')
        print(f'\n【正常/数据不足的检查】共{sum(1 for n,r in rules if not r["触发"])}项')
        for n,r in rules:
            if not r['触发']: print(f'  ○ {n} [{r["_conf"]}]: {r["证据"]}')
        print('\n'+'='*60)
        print('注:基于2D姿态估计,受视角影响;阈值为经验值,应结合专业教练意见。')
        print('='*60)
    return rules

In [ ]:
# 直接修复文件里那一行
with open('/content/drive/MyDrive/ski_app/ski_expert.py', 'r') as f:
    content = f.read()

# 把 rule_flexion 里的 try 变量名替换掉
content = content.replace(
    "yr=np.max(ay)-np.min(ay); apy=np.median(py['顶点']) if py['顶点'] else np.nan; try=np.median(py['过渡']) if py['过渡'] else np.nan",
    "yr=np.max(ay)-np.min(ay); apy=np.median(py['顶点']) if py['顶点'] else np.nan; try_=np.median(py['过渡']) if py['过渡'] else np.nan"
)
content = content.replace(
    "if not(np.isnan(apy)or np.isnan(try)): dn='顶点低/过渡高,方向正确' if apy>try else '顶点未明显低于过渡,屈伸不清晰'",
    "if not(np.isnan(apy)or np.isnan(try_)): dn='顶点低/过渡高,方向正确' if apy>try_ else '顶点未明显低于过渡,屈伸不清晰'"
)

with open('/content/drive/MyDrive/ski_app/ski_expert.py', 'w') as f:
    f.write(content)

print('✅ 已修复 try 变量名')

In [ ]:
import importlib, ski_expert
importlib.reload(ski_expert)
from ski_expert import assess_data_quality, run_expert_system

quality = assess_data_quality(seq, iids, trunk_s, apexes)
rules = run_expert_system(seq, iids, labels, trunk_s, apexes, quality)

In [ ]:
import importlib, ski_expert
importlib.reload(ski_expert)
from ski_expert import assess_data_quality, run_expert_system

quality = assess_data_quality(seq, iids, trunk_s, apexes)
rules = run_expert_system(seq, iids, labels, trunk_s, apexes, quality)

In [ ]:
import glob
tflite = glob.glob('/content/drive/MyDrive/**/*.tflite', recursive=True)
print('TFLite模型:')
for t in tflite: print(' ', t)
import ultralytics; print('\nultralytics版本:', ultralytics.__version__)

In [ ]:
import glob, sys
import numpy as np
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics; importlib.reload(ski_physics)
from ski_physics import extract_all
from ultralytics import YOLO

# ---- 1) 用 TFLite 模型(float32)加载 ----
tflite_path = '/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best_saved_model/best_float32.tflite'
model_tflite = YOLO(tflite_path)
print('✅ TFLite模型加载成功')

# ---- 2) 找一张测试图 ----
imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))
if not imgs:
    # 图片没解压就解压
    import zipfile
    with zipfile.ZipFile('/content/drive/MyDrive/ski2dpose_images_webp.zip') as f:
        f.extractall('/content')
    imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))
test_img = imgs[10]   # 用第10帧
print('测试图:', test_img)

# ---- 3) TFLite 模型预测 ----
res = model_tflite.predict(test_img, imgsz=960, conf=0.5, verbose=False)[0]
print(f'\n检测到 {len(res.keypoints.data)} 个目标')

# ---- 4) 解析输出成 24 关键点(转成 ski_physics 需要的格式)----
if len(res.keypoints.data) == 0:
    print('❌ 没检测到滑手,换张图或调低conf')
else:
    kpts = res.keypoints.data[0].cpu().numpy()   # (24, 3) = [x, y, conf]
    W, H = res.orig_shape[1], res.orig_shape[0]
    # 转成 [x_norm, y_norm, v] 格式(和标注点一致)
    pred_ann = []
    for x, y, c in kpts:
        v = 1 if c >= 0.5 else 0
        pred_ann.append([x/W, y/H, v])
    print(f'解析出 {len(pred_ann)} 个关键点')
    print(f'可见点数(v=1): {sum(1 for p in pred_ann if p[2]==1)}/24')

    # ---- 5) 喂进 ski_physics 算物理量 ----
    r = extract_all(pred_ann)
    print('\n===== TFLite模型 → 物理量(端到端验证)=====')
    for name, (ang, ok) in r['joints'].items():
        if not np.isnan(ang):
            print(f'  {name}: {ang:.1f}°  {"✅" if ok else "⚠️"}')
    print(f"  躯干内倾: {r['trunk_tilt'][0]:+.1f}°  {'✅' if r['trunk_tilt'][1] else '⚠️'}"
          if not np.isnan(r['trunk_tilt'][0]) else "  躯干内倾: 数据不足")
    print(f"  右反弓: {r['angulation_R'][0]:+.1f}°  {'✅' if r['angulation_R'][1] else '⚠️'}"
          if not np.isnan(r['angulation_R'][0]) else "  右反弓: 数据不足")
    cx, cy, mc, ok = r['com']
    print(f"  重心: ({cx:.3f}, {cy:.3f})  覆盖{mc*100:.0f}%  {'✅' if ok else '⚠️'}"
          if not np.isnan(cx) else "  重心: 数据不足")

    print('\n🎉 端到端链路验证:TFLite模型 → 关键点 → 物理量 全部跑通!')

In [ ]:
import glob, sys
import numpy as np
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics; importlib.reload(ski_physics)
from ski_physics import extract_all
from ultralytics import YOLO

# 关键:明确指定 task='pose'(否则它当成detect,没有关键点)
tflite_path = '/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best_saved_model/best_float32.tflite'
model_tflite = YOLO(tflite_path, task='pose')   # ← 加 task='pose'
print('✅ TFLite模型加载成功(pose任务)')

# 找测试图(重启后临时盘可能清空,需重解压)
imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))
if not imgs:
    import zipfile
    with zipfile.ZipFile('/content/drive/MyDrive/ski2dpose_images_webp.zip') as f:
        f.extractall('/content')
    imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))
test_img = imgs[10]
print('测试图:', test_img)

# 预测
res = model_tflite.predict(test_img, imgsz=960, conf=0.5, verbose=False)[0]

# 检查关键点
if res.keypoints is None or len(res.keypoints.data) == 0:
    print('❌ 仍无关键点,贴我看res的内容')
    print('res.boxes:', res.boxes)
else:
    print(f'\n检测到 {len(res.keypoints.data)} 个目标')
    kpts = res.keypoints.data[0].cpu().numpy()
    print(f'关键点形状: {kpts.shape}')   # 应该是 (24, 3)
    W, H = res.orig_shape[1], res.orig_shape[0]
    pred_ann = [[x/W, y/H, (1 if c>=0.5 else 0)] for x,y,c in kpts]
    print(f'可见点数: {sum(1 for p in pred_ann if p[2]==1)}/24')

    # 喂进 ski_physics
    r = extract_all(pred_ann)
    print('\n===== TFLite → 物理量 =====')
    for name,(ang,ok) in r['joints'].items():
        if not np.isnan(ang): print(f'  {name}: {ang:.1f}° {"✅" if ok else "⚠️"}')
    if not np.isnan(r['trunk_tilt'][0]):
        print(f"  躯干内倾: {r['trunk_tilt'][0]:+.1f}° {'✅' if r['trunk_tilt'][1] else '⚠️'}")
    cx,cy,mc,ok = r['com']
    if not np.isnan(cx):
        print(f"  重心: ({cx:.3f},{cy:.3f}) 覆盖{mc*100:.0f}% {'✅' if ok else '⚠️'}")
    print('\n🎉 端到端验证通过:TFLite → 关键点 → 物理量!')

In [ ]:
import glob, sys, json
import numpy as np
from scipy.signal import find_peaks
sys.path.append('/content/drive/MyDrive/ski_app')
import importlib, ski_physics, ski_expert
importlib.reload(ski_physics); importlib.reload(ski_expert)
from ski_physics import extract_all
from ski_expert import assess_data_quality, run_expert_system
from ultralytics import YOLO

# ---- 1) 加载TFLite模型 ----
tflite_path = '/content/drive/MyDrive/ski_app/runs/ski_y26n_v1/weights/best_saved_model/best_float32.tflite'
model = YOLO(tflite_path, task='pose')
print('✅ TFLite模型就绪')

# ---- 2) 逐帧预测那63帧 ----
imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))
if not imgs:
    import zipfile
    with zipfile.ZipFile('/content/drive/MyDrive/ski2dpose_images_webp.zip') as f:
        f.extractall('/content')
    imgs = sorted(glob.glob('/content/**/5UHRvqx1iuQ_0_*.webp', recursive=True))

print(f'开始逐帧预测 {len(imgs)} 帧...')
seq_pred = {}   # 模拟标注格式:{iid: {'annotation': [[x,y,v]*24]}}
iids = []
for idx, img_path in enumerate(imgs):
    iid = f'frame_{idx:05d}'
    iids.append(iid)
    res = model.predict(img_path, imgsz=960, conf=0.5, verbose=False)[0]
    if res.keypoints is None or len(res.keypoints.data)==0:
        # 没检测到:全部标不可见
        ann = [[0,0,0] for _ in range(24)]
    else:
        kpts = res.keypoints.data[0].cpu().numpy()
        W, H = res.orig_shape[1], res.orig_shape[0]
        ann = [[x/W, y/H, (1 if c>=0.5 else 0)] for x,y,c in kpts]
    seq_pred[iid] = {'annotation': ann}
    if idx % 15 == 0: print(f'  已处理 {idx+1}/{len(imgs)} 帧')
print('✅ 逐帧预测完成')

# ---- 3) 重算内倾曲线(用预测点)----
T = len(iids)
trunk_tilt = np.full(T, np.nan)
for i, iid in enumerate(iids):
    r = extract_all(seq_pred[iid]['annotation'])
    if r['trunk_tilt'][1]: trunk_tilt[i] = r['trunk_tilt'][0]
def fill_and_smooth(a, win=5):
    x=np.arange(len(a)); g=~np.isnan(a)
    if g.sum()<2: return a
    return np.convolve(np.interp(x,x[g],a[g]), np.ones(win)/win, mode='same')
trunk_s = fill_and_smooth(trunk_tilt)
print(f'\n内倾曲线: {np.sum(~np.isnan(trunk_tilt))}/{T} 帧可信')

# ---- 4) 转弯识别 + 阶段标签 ----
def detect_turns(curve, prominence=5, distance=5, min_apex_amp=8):
    pk,_=find_peaks(curve,prominence=prominence,distance=distance)
    vl,_=find_peaks(-curve,prominence=prominence,distance=distance)
    ap=[(int(i),+1) for i in pk]+[(int(i),-1) for i in vl]
    ap=[(i,s) for i,s in ap if abs(curve[i])>=min_apex_amp]; ap.sort()
    return ap
apexes = detect_turns(trunk_s)

def label_phases(curve, apexes, flat_thresh=8, apex_window=2):
    T=len(curve); lab=np.array(['过渡']*T,dtype=object); ac=np.abs(curve)
    for af,_ in apexes:
        for j in range(max(0,af-apex_window),min(T,af+apex_window+1)): lab[j]='顶点'
    for i in range(T):
        if lab[i]=='顶点': continue
        if ac[i]<flat_thresh: lab[i]='过渡'; continue
        if i>0: lab[i]='入弯' if ac[i]>ac[i-1] else '出弯'
    return lab
labels = label_phases(trunk_s, apexes)
from collections import Counter
print('阶段分布:', dict(Counter(labels)))

# ---- 5) 跑专家系统(用预测点!)----
quality = assess_data_quality(seq_pred, iids, trunk_s, apexes)
print('\n' + '='*60)
print('  【用TFLite预测点的完整分析报告】')
run_expert_system(seq_pred, iids, labels, trunk_s, apexes, quality)